# 📗 RAG 파이프라인 — 문서 파싱부터 출처 있는 답변까지

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 시간엔 이미 잘 정리된 문장 목록을 임베딩해 검색하고, 찾아온 근거로 답을 쓰게 했습니다. 그런데 실무에서 주어지는 자료는 CSV 가 아니라 **PDF 한 뭉치**입니다. 이번 시간엔 그 PDF 를 **파싱 도구로 읽어 들이는 것부터** 시작해, 자르고 · 색인하고 · 찾아 · **출처가 붙은 답**을 만드는 한 줄기를 끝까지 이어 봅니다.

쓰는 자료는 **개인정보보호위원회가 낸 실제 안내서 PDF** 두 건입니다. 진짜 문서라서 쪽 번호가 진짜이고, 답에 붙인 출처를 **PDF 뷰어에서 그 쪽으로 열어 직접 대조**할 수 있습니다.

## ⏪ 복습 — 지난 시간까지 쌓은 재료

- **임베딩**: `SentenceTransformer('jhgan/ko-sroberta-multitask')` 로 문장을 768개의 숫자로 바꿨습니다.
- **벡터 DB**: `chromadb` 컬렉션에 벡터를 넣고 질문과 가까운 것을 꺼내 봤습니다. 메타데이터로 **조건을 걸어 검색**하는 것도 해 봤습니다.
- **구조화된 출력**: `pydantic` 클래스를 `client.chat.completions.parse(response_format=...)` 에 넘겨, 자유 문장에서 **정해진 필드**를 뽑아냈습니다.
- **LLM 으로 답 쓰기**: 찾아온 근거를 프롬프트에 실어 답을 만들게 했습니다.

오늘은 이 넷을 **하나의 파이프라인**으로 잇습니다. 앞에 **PDF 파싱**과 **청킹**이 새로 붙고, 뒤에 **출처 표기**가 붙습니다.

**오늘의 목표**

- [ ] `pymupdf4llm` 으로 PDF 를 **마크다운**으로 읽고, `page_chunks=True` 로 **쪽 번호까지** 함께 받는다.
- [ ] 같은 쪽을 `page.get_text()` 로도 뽑아 **무엇이 남고 무엇이 사라지는지** 실측으로 비교하고, 어느 도구를 쓸지 스스로 고른다.
- [ ] 문서를 통째로 임베딩하면 왜 검색이 무뎌지는지 재 보고, **세 가지 청킹 전략**을 만들어 비교한다.
- [ ] 청크를 임베딩해 **디스크에 남는 색인**을 만들고, 메타데이터에 문서명·발간연도·쪽을 싣는다.
- [ ] 검색 결과를 **문서명·쪽·거리**와 함께 읽는다.
- [ ] 질문에서 **메타데이터 필터를 자동으로 뽑아** 검색 범위를 좁힌다.
- [ ] 찾은 근거로 답을 만들고 **문서명과 쪽 번호가 붙은 출처**를 함께 낸다.

아래 두 셀을 먼저 실행해 OpenAI 클라이언트와 한국어 임베딩 모델을 준비하세요. 따라하기까지 모두 풀면 이 노트북은 **실제 OpenAI 호출을 9회** 합니다(필터 추출 4회, 답 생성 5회).

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 이 셀은 실행만 하세요.
# 15일차와 같은 방식입니다: .env 의 OPENAI_API_KEY 로 실제 OpenAI 에 연결합니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인한다 — OpenAI() 를 만든 뒤에 검사하면 SDK 인증 오류가 먼저 나서 이 안내가 묻힌다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from openai import OpenAI

client = OpenAI()
print("OpenAI 클라이언트 준비 완료 — 실제 API 연결됨")

In [ ]:
# 이 노트북에서 계속 쓸 라이브러리를 한 번에 불러옵니다.
import time

import numpy as np
import pandas as pd
import pymupdf                 # PDF 를 여는 저수준 도구
import pymupdf4llm             # 같은 PDF 를 '마크다운'으로 바꿔 주는 도구

print("준비 완료")

In [ ]:
# [제공 코드] 임베딩 모델 준비 — 지난 시간에 쓴 한국어 문장 임베딩 모델입니다(불러오는 데 잠시 걸립니다).
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 (768차원)')

## 1. PDF 를 파싱 도구로 읽는다

본격적으로 재 보기 전에, **오늘 쓸 파싱 도구 네 가지가 무엇이고 어떻게 다른지** 먼저 정리합니다.

| 도구 | 어떤 도구인가 | 특징 | OCR | 무료 |
|---|---|---|---|---|
| **PyMuPDF `get_text()`** | 로컬 라이브러리 | 글자 층을 그대로 뽑는다. 가장 빠르다. **구조는 남지 않는다** | ✕ | 오픈소스 — 단 **AGPL** |
| **`pymupdf4llm.to_markdown()`** | 같은 라이브러리의 마크다운 변환 | 제목·표를 마크다운 구조로 살린다. `page_chunks=True` 로 쪽 번호까지 | ✕ | 오픈소스 — 단 **AGPL** |
| **Upstage Document Parse** | 클라우드 API(국내) | 레이아웃을 알아보고 제목·문단·표·그림을 **원소로 분리**. 원소마다 쪽 번호 | ○ | **가입 크레딧만** — 상시 무료 없음, 이후 쪽당 과금 |
| **LlamaParse** | 클라우드 API(해외) | **티어를 골라 품질과 단가를 맞바꾼다**(fast · cost effective · agentic · agentic plus) | ○ | **무료 플랜 있음** — 주기마다 크레딧이 갱신됨 |

**OCR 열이 뜻하는 것.** 로컬 두 도구는 **이미 문서 안에 들어 있는 글자**를 꺼내 올 뿐입니다. 스캔한 종이처럼 글자 층이 없으면 **한 글자도 뽑지 못합니다.** 클라우드 두 도구는 **이미지에서 글자를 읽어 냅니다.**

**"무료" 의 조건이 서로 다릅니다.** 로컬 두 도구는 같은 라이브러리이고 라이선스가 이렇게 적혀 있습니다.

> `pymupdf` · `pymupdf4llm` — *Dual Licensed - GNU AFFERO GPL 3.0 or Artifex Commercial License*

공부하거나 사내에서 쓰는 데는 무료입니다. 그런데 이 라이브러리를 넣은 서비스를 **외부에 배포**하면 **AGPL 이 소스 공개를 요구**합니다. 그게 곤란한 회사는 상용 라이선스를 사거나 다른 도구를 씁니다. **클라우드 API 는 그 대신 돈을 내고 이 문제를 피합니다** — 도구를 고를 때 실제로 갈리는 지점은 가격표가 아니라 **배포할 때 소스를 공개해야 하는가**입니다.

요금과 무료 한도는 자주 바뀌니 교재에 적지 않습니다. **각 서비스 콘솔에서 직접 확인하세요.**

**공식 문서와 키 발급**

- **PyMuPDF** — 문서 <https://pymupdf.readthedocs.io> · 키 불필요
- **pymupdf4llm** — 문서 <https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/> · 키 불필요
- **Upstage Document Parse** — 문서 <https://console.upstage.ai/docs/capabilities/document-digitization/document-parsing> · 제품 소개 <https://www.upstage.ai/ko/products/document-parse> · **키 발급** <https://console.upstage.ai>
- **LlamaParse** — 문서 <https://developers.llamaindex.ai/llamaparse/> · 요금 <https://developers.llamaindex.ai/llamaparse/general/pricing/> · **키 발급** <https://cloud.llamaindex.ai>

> **표만 보고 고르지 말고, 아래에서 직접 재 본 뒤 다시 이 표로 돌아오세요.** 이 표는 전체를 한눈에 보기 위한 것이고, 고르는 기준은 §1 을 다 보고 나서 세웁니다.

### 문서 파싱은 왜 필요할까요?

RAG 의 첫 단계는 언제나 **문서를 텍스트로 만드는 일**입니다. 이 단계가 엉망이면 뒤의 임베딩·검색·생성이 아무리 좋아도 소용이 없습니다. 쓰레기를 넣으면 쓰레기가 나옵니다.

그런데 PDF 안에는 "이 글자를 이 좌표에 이 크기로 찍어라" 라는 지시만 들어 있고, '여기부터 제목', '이건 표의 한 칸' 같은 말은 없습니다. 그래서 파싱 도구가 하는 일은 **흩어진 글자를 주워 담고, 크기·굵기·칸의 위치를 보고 구조를 되짚어 주는 것**입니다.

위 표의 넷 가운데 **로컬 두 도구**부터 손에 익힙니다. 부르는 법은 이렇습니다.

| 도구 | 부르는 법 | 돌려주는 것 |
|---|---|---|
| `pymupdf4llm` | `to_markdown(경로)` | 문서 전체를 **마크다운 문자열** 하나로 |
| `pymupdf4llm` | `to_markdown(경로, page_chunks=True)` | **쪽 단위 딕셔너리 목록**(본문 + 메타데이터) |
| `pymupdf` | `pymupdf.open(경로)[i].get_text()` | 그 쪽의 **글자만** 이어 붙인 문자열 |

먼저 가장 단순한 한 줄부터 봅니다.

In [ ]:
PDF_AI_PATH = 'data/원본/생성형AI_개인정보_처리_안내서.pdf'

# 한 줄이면 PDF 전체가 마크다운 문자열 하나로 나온다.
t0 = time.perf_counter()
# show_progress=True 면 진행 막대가 나온다 -- 오래 걸리는 문서에서 멈춘 게 아님을 알 수 있다.
md_all = pymupdf4llm.to_markdown(PDF_AI_PATH, show_progress=True)
md_seconds = time.perf_counter() - t0

print(f"\n글자 수 {len(md_all):,}자 · 걸린 시간 {md_seconds:.1f}초")
print("\n----- 앞부분 260자 -----")
print(md_all[:260])

글자는 잘 나왔습니다. 그런데 **한 덩어리**라서 문제가 하나 있습니다 — 어느 대목이 **몇 쪽에서 왔는지** 알 수 없습니다. 나중에 답에 출처를 붙이려면 그 쪽 번호가 꼭 필요합니다.

그래서 **`page_chunks=True`** 를 씁니다. 이러면 문자열 하나가 아니라 **쪽마다 딕셔너리 하나**인 목록이 돌아옵니다.

> ⚠️ 쪽 번호가 들어 있는 키 이름은 `page` 가 아니라 **`metadata['page_number']`** 입니다. 이름을 헷갈리면 `KeyError` 가 납니다.

In [ ]:
pages = pymupdf4llm.to_markdown(PDF_AI_PATH, page_chunks=True, show_progress=True)

print(f"쪽 개수: {len(pages)}")
print("쪽 하나에 들어 있는 키:", list(pages[0].keys()))
print("metadata 안의 키:", list(pages[0]["metadata"].keys()))

# 쪽 번호는 metadata 안에 있다 -> 'page' 가 아니라 'page_number'
print("\n앞 5쪽의 번호:", [p["metadata"]["page_number"] for p in pages[:5]])

이제 쪽 하나를 골라 **무엇이 나왔는지** 직접 봅니다. 9쪽은 **표가 들어 있는 쪽**입니다.

In [ ]:
# 쪽 번호로 찾는다 -- 목록의 순서가 아니라 metadata 의 번호를 기준으로 삼아야 안전하다.
page9 = [p for p in pages if p['metadata']['page_number'] == 9][0]

print(f"9쪽 글자 수: {len(page9['text']):,}자")
print("\n----- 앞부분 700자 -----")
print(page9['text'][:700])

`#` 로 시작하는 **제목 줄**과 `|` 로 칸을 나눈 **표**가 그대로 살아 있습니다. 마크다운은 사람도 읽고 LLM 도 잘 읽는 표기라, 이 상태 그대로 근거로 써도 됩니다.

그럼 **그냥 텍스트로 뽑으면** 어떻게 다를까요. 같은 9쪽을 `pymupdf` 의 `get_text()` 로 꺼내 봅니다.

In [ ]:
doc = pymupdf.open(PDF_AI_PATH)

# get_text 는 0부터 세는 색인이라 9쪽은 doc[8] 이다.
raw9 = doc[8].get_text()

print(f"9쪽 글자 수: {len(raw9):,}자")
print("\n----- 앞부분 700자 -----")
print(raw9[:700])

표가 **한 줄에 한 칸씩 세로로 풀려** 버렸습니다. `구 분` 과 `주요 내용` 이 어느 행에 짝지어 있었는지가 사라져서, 이 글만 읽고는 "어느 사건이 몇 년에 있었나"를 되짚기 어렵습니다. 제목도 그냥 한 줄의 글이 되어 **어디까지가 한 항목인지** 알 수 없습니다.

느낌이 아니라 **숫자로** 확인해 봅시다. 쪽 몇 개를 골라 두 방식의 결과에서 **제목 줄 수·표 줄 수·글자 수**를 세어 나란히 놓고, 같은 PDF 전체를 두 방식으로 읽는 **시간**도 함께 잽니다.

In [ ]:
def count_marks(text):
    """마크다운 구조 흔적을 센다: '#' 로 시작하는 제목 줄과 '|' 로 시작하는 표 줄."""
    lines = [line.strip() for line in text.splitlines()]
    heads = sum(1 for line in lines if line.startswith('#'))
    rows = sum(1 for line in lines if line.startswith('|'))
    return heads, rows

# 쪽 네 개를 골라 두 방식의 결과를 나란히 센다.
rows = []
for page_no in [9, 14, 20, 27]:
    md_text = [p for p in pages if p['metadata']['page_number'] == page_no][0]['text']
    raw_text = doc[page_no - 1].get_text()
    md_heads, md_rows = count_marks(md_text)
    raw_heads, raw_rows = count_marks(raw_text)
    rows.append({'쪽': page_no,
                 'md_제목줄': md_heads, 'md_표줄': md_rows, 'md_글자수': len(md_text),
                 'raw_제목줄': raw_heads, 'raw_표줄': raw_rows, 'raw_글자수': len(raw_text)})

display(pd.DataFrame(rows))

# 구조만 보지 말고 값도 보자 -- 같은 PDF 전체를 두 방식으로 읽어 시간을 잰다.
t0 = time.perf_counter()
raw_all = '\n'.join(page.get_text() for page in doc)
raw_seconds = time.perf_counter() - t0

# md_seconds 는 앞 셀에서 잰 to_markdown 시간이다.
print(f"\nget_text    {raw_seconds:6.2f}초 · {len(raw_all):,}자")
print(f"to_markdown {md_seconds:6.2f}초 · {len(md_all):,}자")

구조는 마크다운 쪽이 훨씬 많이 살아남지만, **시간은 `get_text` 가 압도적으로 빠릅니다.** 몇 배인지는 기기마다 다르니 숫자를 외우지 말고 **자릿수가 다르다**는 것만 기억하세요. 이 둘이 곧 도구를 고르는 두 축입니다.

### 다 읽지 않아도 된다 — `pages` 로 필요한 쪽만

위에서 52쪽짜리 안내서 하나를 마크다운으로 바꾸는 데 몇 초가 들었습니다. 실무 문서는 수백 쪽인 경우가 흔한데, **정작 필요한 건 몇 쪽뿐**일 때가 많습니다. `to_markdown` 은 읽을 쪽을 지정하는 **`pages`** 인자를 받습니다.

그런데 여기에 **자주 사고를 내는 함정**이 하나 있습니다.

| 값 | 세는 방식 |
|---|---|
| `pages=[...]` 에 넣는 번호 | **0부터** 세는 색인 (파이썬 목록과 같다) |
| `metadata['page_number']` | **1부터** 세는 쪽 번호 (PDF 뷰어에 보이는 그 번호) |

기준이 서로 다르니 **넣은 번호와 돌아온 번호가 한 칸씩 어긋납니다.** 이걸 모르고 인용하면 출처가 통째로 한 쪽씩 밀립니다. 말로 외우지 말고 직접 확인합시다.

In [ ]:
# 세 쪽만 골라 읽는다. 넣은 번호와 돌아온 번호를 나란히 놓고 본다.
want = [12, 13, 14]

t0 = time.perf_counter()
picked_pages = pymupdf4llm.to_markdown(PDF_AI_PATH, pages=want, page_chunks=True, show_progress=True)
picked_seconds = time.perf_counter() - t0

got = [p['metadata']['page_number'] for p in picked_pages]
print(f"pages 에 넣은 번호 : {want}")
print(f"page_number 로 온 번호: {got}")

# md_seconds 는 같은 PDF 를 통째로 읽었을 때의 시간이다.
print(f"\n3쪽만 읽기 {picked_seconds:.2f}초  vs  52쪽 통째 {md_seconds:.2f}초")

넣은 번호보다 **1 큰 쪽**이 돌아옵니다. 그러니 **뷰어에서 본 15쪽을 읽고 싶으면 `pages=[14]`** 를 넘겨야 하고, 출처로 쓸 번호는 **돌아온 `page_number` 쪽**입니다. 헷갈릴 때는 지금처럼 넣은 값과 나온 값을 한 번 찍어 보면 끝납니다.

### 그래서 어느 문서에 어느 도구를 쓸까요?

방금 잰 두 가지 — **구조가 얼마나 남는가**와 **얼마나 걸리는가** — 가 그대로 판단 기준이 됩니다.

| 문서가 이렇다면 | 고를 도구 | 근거 |
|---|---|---|
| **표·제목이 뜻을 나르는** 문서(안내서·보고서·규정집) | `to_markdown` | 구조가 사라지면 표의 행-열 짝이 끊긴다 |
| **줄글만 있는** 문서(계약서 본문·기사·소설) | `get_text` | 살릴 구조가 없는데 시간만 더 든다 |
| **쪽 번호를 출처로 붙여야** 한다 | `to_markdown(page_chunks=True)` | `metadata['page_number']` 가 함께 온다 |
| 문서가 **수만 쪽**이고 한 번에 훑어야 한다 | `get_text` 먼저 | 방금 본 것처럼 시간 차가 자릿수로 벌어진다 |
| 필요한 대목이 **몇 쪽뿐**이다 | `to_markdown(pages=[...])` | 전체를 읽지 않아 시간이 크게 줄어든다 |
| **스캔한 종이**를 이미지로 담은 PDF · 표가 여러 겹으로 **병합된** 문서 | 로컬 도구로는 안 된다 → **상용 파서**(Upstage Document Parse · LlamaParse) | 글자 층이 없으면 뽑을 것이 없고, 병합된 표는 열이 뭉개진다. 대신 **쪽 단위 과금**·네트워크·키가 든다 |

이 가운데 실무에서 가장 먼저 부딪히는 것은 **스캔본**입니다. 파일을 열자마자 **글자 층이 있는지부터** 봅니다 — `len(doc[0].get_text().strip()) < 50` 이면 글자가 없는 스캔본이라 지금까지 쓴 두 도구로는 아무것도 못 뽑습니다. 오늘 쓰는 안내서들은 글자 층이 있어 그냥 읽힙니다.

그럼 **글자 층이 없거나 표가 복잡한 문서는 어떻게 할까요?** 표의 마지막 줄에 적은 대로 상용 파서를 씁니다. 바로 이어서 Upstage 와 LlamaParse 를 직접 호출해 봅니다.

<img src="images/파싱_두갈래.png" width="900">

### 로컬 도구로 안 되는 문서 — 상용 파서 써 보기

위 표의 마지막 줄에 적은 두 경우를 여기서 다룹니다. 스캔한 종이는 글자 층이 없어 `get_text` 도 `to_markdown` 도 한 글자도 뽑지 못합니다. 표가 여러 겹으로 병합된 한국어 문서도 로컬 도구가 자주 놓칩니다. 이럴 때 쓰는 것이 **문서 파싱을 서비스로 파는 상용 API** 입니다.

여기서는 국내 서비스인 **Upstage Document Parse** 를 씁니다. 문서를 올리면 레이아웃을 알아보고 **제목·문단·표·그림을 원소(element)로 나눠** 돌려줍니다. 스캔본이면 OCR 까지 알아서 합니다.

| 돌려주는 것 | 내용 |
|---|---|
| `elements[]` | 원소 목록. 각 원소에 `category`(`paragraph`·`table`·`figure`·`heading1` …)와 **`page`**, `content.{text,html,markdown}` |
| `content` | 문서 전체를 이어 붙인 `text`·`html`·`markdown` |
| `usage.pages` | **이번 호출이 몇 쪽으로 과금되는지** |

> **`elements[].page` 는 1부터 셉니다.** 앞에서 본 `metadata['page_number']` 와 **같은 규약**이라 우리 파이프라인에 그대로 꽂을 수 있습니다(`pages=` 인자만 0부터였다는 것을 기억하세요).

**키를 발급받으세요.** 이 실습은 키가 있어야 진행됩니다 — 없으면 아래 셀이 안내와 함께 **멈춥니다.**

1. <https://console.upstage.ai> 에 가입합니다(Google·GitHub 계정으로도 됩니다).
2. 콘솔의 **API Keys** 에서 키를 발급합니다(`up_` 으로 시작합니다).
3. 이 폴더의 `.env` 에 한 줄 추가합니다 — `UPSTAGE_API_KEY=up_...` · 그리고 **커널을 재시작**합니다.

- 공식 문서: <https://console.upstage.ai/docs/capabilities/document-digitization/document-parsing>
- 제품 소개: <https://www.upstage.ai/ko/products/document-parse>

**비용은 쪽 단위로 매겨집니다.** 새로 가입하면 시험해 볼 크레딧이 주어집니다. 요금과 남은 크레딧은 바뀌니 **콘솔에서 직접 확인**하세요. 교재는 금액을 적지 않는 대신, 호출할 때마다 **`usage.pages` 를 찍어** 이번 호출이 몇 쪽으로 계산됐는지 여러분이 직접 보게 합니다.

그래서 **문서를 통째로 보내지 않습니다.** 표가 있는 두 쪽만 잘라 임시 PDF 로 만들어 보냅니다 — 앞에서 배운 `pages` 개념의 실무 응용입니다. (동기 방식은 100쪽까지 받습니다.)

> **결과를 미리 정해 두고 보지 마세요.** 오늘 쓰는 이 문서는 글자 층이 온전해서 로컬 도구도 표를 잡습니다 — 다만 **얼마나 정확히 잡는지는 표마다 다릅니다.** 두 결과를 나란히 놓고 **이 문서에 상용 파서가 필요한지**를 직접 판단하는 것이 이 셀의 목적입니다. 돈이 드는 도구는 **그 돈을 낼 만한 문서**에만 씁니다.

**무엇을 볼까요.** 표가 제대로 뽑혔는지는 **열이 몇 칸으로 잡혔는지**를 보면 가장 빠릅니다. 원본을 PDF 뷰어로 열어 놓고 그 칸 수와 맞는지 대 보세요.

In [ ]:
import json
from pathlib import Path

# 처리방침 표준안에서 표가 많은 두 쪽만 골라 작은 PDF 를 만든다 -- 통째로 보내면 그만큼 과금된다.
# 이 파일은 지우지 않고 output/ 에 남긴다: 네 도구에 넣은 '입력' 을 결과와 나란히 열어 볼 수 있어야 한다.
PDF_PP_PATH = 'data/원본/개인정보_처리방침_표준안.pdf'
WANT_PAGES = [9, 32]        # 뷰어에 보이는 쪽 번호(1부터)

OUT_DIR = Path('output' if Path('data').exists() else '../output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_PDF_PATH = OUT_DIR / '파싱_0_입력_2쪽.pdf'

source = pymupdf.open(PDF_PP_PATH)
picked = pymupdf.open()                      # 빈 PDF 를 만들어 고른 쪽만 옮겨 담는다
for page_no in WANT_PAGES:
    # insert_pdf 의 쪽 번호는 0부터다 -- 뷰어 번호에서 1을 뺀다.
    picked.insert_pdf(source, from_page=page_no - 1, to_page=page_no - 1)
picked.save(SAMPLE_PDF_PATH)

print(f'저장: {SAMPLE_PDF_PATH} ({picked.page_count}쪽 · {SAMPLE_PDF_PATH.stat().st_size / 1024:.0f}KB)')
print('-> 이 파일 안에서는 쪽 번호가 1부터 다시 매겨진다. 출처로 쓸 원래 번호는 따로 챙겨야 한다.')

def save_parsed(filename, text):
    """파싱 결과를 output/ 에 남기고 경로를 알려 준다. 화면 출력은 잘려서 비교가 안 되기 때문이다."""
    path = OUT_DIR / filename
    path.write_text(text, encoding='utf-8')
    print(f'저장: {path} ({len(text):,}자)')
    return path

# 로컬 두 도구의 결과를 같은 임시 PDF 로 뽑아 파일로 남긴다 -- 나중에 넷을 나란히 열어 보려면 대상이 같아야 한다.
save_parsed('파싱_1_get_text.txt', '\n'.join(page.get_text() for page in picked))
save_parsed('파싱_2_to_markdown.md', pymupdf4llm.to_markdown(str(SAMPLE_PDF_PATH), show_progress=True))

In [ ]:
import os

import requests

upstage_key = os.getenv('UPSTAGE_API_KEY')

# 키가 없으면 여기서 멈춘다 -- 조용히 건너뛰면 무엇이 빠졌는지 모른 채 수업이 진행된다.
if not upstage_key:
    raise RuntimeError(
        '이 실습은 Upstage API 키가 필요합니다 -- UPSTAGE_API_KEY 를 찾지 못했습니다.\n'
        '  1) https://console.upstage.ai 가입 후 API Keys 에서 발급\n'
        '  2) 일차 폴더의 .env 에  UPSTAGE_API_KEY=up_...  추가\n'
        '  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요')

response = requests.post(
    'https://api.upstage.ai/v1/document-digitization',
    headers={'Authorization': f'Bearer {upstage_key}'},
    files={'document': open(SAMPLE_PDF_PATH, 'rb')},
    # 값은 모두 문자열로 넘긴다. coordinates 는 좌표라 지금은 필요 없어 끈다.
    data={'model': 'document-parse',
          'output_formats': '["markdown"]',
          'coordinates': 'false'},
    timeout=120)

if response.status_code != 200:
    # 실패도 눈에 보이게 -- 메시지만 찍고 계속 가면 크레딧 소진·키 오류를 모르고 지나친다.
    raise RuntimeError(
        f'Upstage 요청 실패({response.status_code}). 키와 잔여 크레딧을 콘솔에서 확인하세요.\n'
        f'{response.text[:300]}')

parsed = response.json()

# 이번 호출이 몇 쪽으로 과금되는지 -- 금액이 아니라 이 숫자를 습관적으로 확인한다.
print(f"과금 쪽 수 usage.pages: {parsed['usage']['pages']}")

# 어떤 원소로 나눠 왔는지 세어 본다.
counts = {}
for element in parsed['elements']:
    counts[element['category']] = counts.get(element['category'], 0) + 1
print('원소 종류:', counts)

**표가 몇 칸짜리로 잡혔는지** 세어, 로컬 도구와 견줍니다. 열이 사라졌는지 보려면 이 숫자가 가장 빠릅니다.

In [ ]:
# 마크다운에서 표마다 머리 줄을 하나씩 모은다.
# 한 쪽에 표가 둘 이상일 수 있다 -- 그래서 '쪽마다 첫 | 줄' 로 세면 뒤 표를 놓친다.
# 연속으로 이어지는 | 줄 덩어리를 표 하나로 본다.
def table_heads(text):
    heads, in_table = [], False
    for line in text.splitlines():
        stripped = line.strip()
        if stripped.startswith('|'):
            if not in_table:
                heads.append(stripped)
            in_table = True
        else:
            in_table = False
    return heads

# '|a|b|c|' 의 칸 수. 양 끝 | 를 하나씩만 떼고 나눈다.
# strip('|') 을 쓰면 안 된다 -- 그건 양끝의 | 를 몇 개든 떼어 내서,
# 끝이 '||' 인 행(마지막 칸이 빈 행)에서 빈 칸이 통째로 사라진다.
def count_cols(line):
    if not line:
        return 0
    if line.startswith('|'):
        line = line[1:]
    if line.endswith('|'):
        line = line[:-1]
    return len(line.split('|'))

# Upstage 는 elements 가 이미 표 단위라 원소마다 머리 줄 하나만 본다.
tables = [e for e in parsed['elements'] if e['category'] == 'table']
up_heads = [h for t in tables for h in table_heads(t['content']['markdown'])[:1]]
up_cols = [count_cols(h) for h in up_heads]

# 로컬 도구 결과는 한 쪽에 표가 여럿 섞여 나오므로 덩어리 단위로 센다.
local = pymupdf4llm.to_markdown(str(SAMPLE_PDF_PATH), page_chunks=True, show_progress=True)
local_heads = [h for page in local for h in table_heads(page['text'])]
local_cols = [count_cols(h) for h in local_heads]

print(f'표별 열 수  Upstage: {up_cols}  |  pymupdf4llm: {local_cols}')

숫자만 보고는 어느 쪽이 맞는지 알 수 없으니, **첫 쪽 표의 머리 줄**을 나란히 놓고 눈으로 봅니다.

In [ ]:
# 첫 쪽의 같은 표를 두 도구가 어떻게 잡았는지 머리 줄만 나란히 놓는다.
first_up = next((h for h, t in zip(up_heads, tables) if t['page'] == 1 and h), '')
print('----- 첫 쪽 표의 머리 줄 -----')
print(f'Upstage     : {first_up}')
print(f'pymupdf4llm : {local_heads[0]}')

# 다음 셀에서 LlamaParse 와 나란히 놓으려고 첫 표만 따로 챙겨 둔다.
up_first_table = next((t['content']['markdown'] for t in tables if t['page'] == 1), '')

# 결과와 응답 원본을 남긴다. JSON 을 열어 보면 '원소로 나눠 준다' 는 말이 눈에 들어온다.
save_parsed('파싱_3_upstage.md', parsed['content']['markdown'])
save_parsed('파싱_3_upstage.json', json.dumps(parsed, ensure_ascii=False, indent=2))

# 임시 PDF 는 다음 셀에서 LlamaParse 에도 보내야 하니 아직 지우지 않는다.

**두 도구 다 표를 세 개 찾았습니다. 갈리는 건 그중 하나의 열 수입니다.**

숫자만 보고는 어느 쪽이 맞는지 알 수 없습니다. 그래서 **원본 PDF 를 열어** 세 표의 칸 수를 직접 세어 대 봤습니다.

| 표 | 원본 | Upstage | `pymupdf4llm` |
|---|---|---|---|
| 9쪽 표 | **4열** | 4열 (일치) | **3열 — 한 열이 사라짐** |
| 32쪽 표1 | 5열 | 5열 (일치) | 5열 (일치) |
| 32쪽 표2 | 3열 | 3열 (일치) | 3열 (일치) |

머리 줄을 보면 9쪽에서 무슨 일이 있었는지 바로 보입니다 — 로컬 도구가 **`표준(안)` 과 `비고` 를 한 칸에 `<br>` 로 붙여** 버렸습니다. 원본에서 그 두 열은 13행부터 셀이 병합돼 있는데, 그 병합을 **두 열이 원래 하나였던 것으로** 읽은 것입니다.

**행은 살아 있는데 열이 죽었다는 게 무섭습니다.** 22개 행이 다 나와 있어 눈으로 훑으면 멀쩡해 보입니다. 그런데 이 표를 근거로 “13번 항목의 비고가 무엇이냐”고 물으면 **답할 방법이 없습니다.** 뒤에서 배울 검색·생성이 아무리 좋아도 소용없습니다 — 그 정보가 애초에 색인에 들어가지 않았기 때문입니다.

**그래서 이 문서에서는 Upstage 가 나았습니다.** 세 표를 모두 원본대로 잡았고, 로컬 도구는 한 표에서 열 하나를 잃었습니다. 실측이 그렇게 말하니 그대로 받아들입니다.

다만 이건 **문서 두 쪽으로 잰 결과**입니다. 나머지 두 표는 로컬 도구도 똑같이 잡았습니다. 그러니 “비싼 도구가 늘 낫다”로 옮겨 적지 말고, 이렇게 기억하세요.

> **내 문서로 재 보고 고른다. 특히 표가 답의 근거라면, 뽑아낸 표의 열 수를 원본과 대조한다.**

이것이 §1 이 처음부터 이야기해 온 **도구를 고르는 기준**의 마지막 조각입니다. 무료·유료를 미리 정해 두는 것이 아니라, **내 문서에서 무엇이 살아남고 무엇이 사라지는지 재 보고** 정합니다.

> **함정 하나 더.** 표를 셀 때 **쪽마다 하나**라고 가정하면 32쪽의 둘째 표를 놓칩니다. 한 쪽에 표가 여럿일 수 있으니, 세는 단위는 **쪽이 아니라 표 덩어리**여야 합니다.

### 상용 파서가 하나뿐일까요 — LlamaParse 도 같은 표로

상용 파서는 여럿입니다. 맨 앞 표에 있던 **LlamaParse** 로 **같은 임시 PDF** 를 한 번 더 파싱해, 셋을 나란히 놓아 봅니다. 먼저 표를 세는 데 쓸 도구부터 준비합니다.

> 키는 <https://cloud.llamaindex.ai> 에서 발급해 `.env` 에 `LLAMA_CLOUD_API_KEY=llx-...` 로 넣습니다. 없으면 아래 셀이 멈춥니다.

In [ ]:
import re

# 표 한 줄을 칸 목록으로 나눈다.
def split_cells(line, greedy=False):
    # greedy=True 는 흔히 저지르는 실수다 -- str.strip('|') 은 양끝의 | 를 몇 개든 떼어 내서,
    # 끝이 '||' 인 행에서는 마지막 빈 칸이 통째로 사라진다. 마크다운 규칙은 양끝 하나씩이다.
    if greedy:
        line = line.strip('|')
    else:
        if line.startswith('|'):
            line = line[1:]
        if line.endswith('|'):
            line = line[:-1]
    return [c.strip() for c in line.split('|')]

# 마크다운 파이프 표든 HTML <table> 이든 [[셀, ...], ...] 로 바꾼다.
# 도구마다 표를 주는 형식이 달라서, 세기 전에 한 모양으로 맞춰 놓아야 한다.
def table_rows(text):
    if '<table' in text.lower():
        rows = []
        for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', text, re.S | re.I):
            cells = re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', tr, re.S | re.I)
            rows.append([re.sub(r'<[^>]+>', '', c).strip() for c in cells])
        return [r for r in rows if r]

    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith('|'):
            continue
        cells = split_cells(line)
        # |---|---| 같은 구분선은 내용이 아니다.
        if all(set(c) <= set('-: ') for c in cells):
            continue
        rows.append(cells)
    return rows

# 표 하나를 두 숫자로 요약한다: 열·데이터 행.
def describe(name, text):
    rows = table_rows(text)
    if not rows:
        print(f'{name:12} 표를 찾지 못했습니다')
        return

    width = len(rows[0])          # 머리 줄의 칸 수 = 그 표의 열 수
    body = rows[1:]               # 첫 줄은 머리 줄이라 데이터 행에서 뺀다

    print(f'{name:12} 열 {width} · 데이터 행 {len(body)}')

print('표 비교 도구 준비 완료')

**여기부터 세 단계입니다** — 1) 파일 업로드 → 2) 파싱 작업 생성 → 3) 끝날 때까지 상태 확인. 단계마다 셀을 나눠, 각 단계가 실제로 무엇을 돌려주는지 눈으로 확인하며 진행합니다.

In [ ]:
llama_key = os.getenv('LLAMA_CLOUD_API_KEY')
LLAMA_BASE = 'https://api.cloud.llamaindex.ai'

# 키가 없으면 여기서 멈춘다 -- 우회로를 두면 무엇이 빠졌는지 모른 채 지나가게 된다.
if not llama_key:
    raise RuntimeError(
        '이 실습은 LlamaParse API 키가 필요합니다 -- LLAMA_CLOUD_API_KEY 를 찾지 못했습니다.\n'
        '  1) https://cloud.llamaindex.ai 가입 후 API Keys 에서 발급\n'
        '  2) 일차 폴더의 .env 에  LLAMA_CLOUD_API_KEY=llx-...  추가\n'
        '  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요')

head = {'Authorization': f'Bearer {llama_key}'}

# 1) 파일을 먼저 올린다.
with open(SAMPLE_PDF_PATH, 'rb') as fp:
    up = requests.post(f'{LLAMA_BASE}/api/v1/beta/files', headers=head,
                       files={'file': (SAMPLE_PDF_PATH.name, fp, 'application/pdf')},
                       data={'purpose': 'parse'}, timeout=120)
file_id = up.json()['id']
print(f'업로드 완료 -> file_id: {file_id}')

In [ ]:
# 2) 그 파일로 파싱 작업을 만든다. tier 로 품질과 단가를 맞바꾼다.
job = requests.post(f'{LLAMA_BASE}/api/v2/parse', headers=head,
                    json={'file_id': file_id, 'tier': 'cost_effective', 'version': 'latest'},
                    timeout=120).json()
job_id = job['id']
print(f'작업 생성 완료 -> job_id: {job_id}')

In [ ]:
# 3) 끝날 때까지 물어본다.
#    ※ 함정: 응답 맨 위에는 status 가 없다. 상태는 job['status'] 안에 있다 -- 맨 위를 보면 영영 안 끝난다.
result = {}
for _ in range(20):
    result = requests.get(f'{LLAMA_BASE}/api/v2/parse/{job_id}?expand=markdown',
                          headers=head, timeout=120).json()
    if result.get('job', {}).get('status') in ('COMPLETED', 'ERROR', 'CANCELED'):
        break
    time.sleep(3)

status = result.get('job', {}).get('status')
print(f'작업 상태: {status}')

# 끝나지 않았거나 실패했으면 여기서 멈춘다 -- 빈 결과로 다음 단계를 계속하지 않는다.
if status != 'COMPLETED':
    raise RuntimeError(f'LlamaParse 작업이 끝나지 않았습니다(상태: {status}).')

작업이 끝났으니 **결과를 파일로 남기고**, 첫 쪽 마크다운을 뒤 비교에 쓸 이름(`llama_first`)으로 꺼내 둡니다.

In [ ]:
# 이름을 llama_pages 로 둔다 -- 앞 절에서 만든 pages(쪽 목록)를 덮어쓰면 뒤 셀이 깨진다.
llama_pages = result['markdown']['pages']
llama_first = llama_pages[0]['markdown']
print(f"받은 쪽 수: {len(llama_pages)} · 첫 쪽 번호: {llama_pages[0]['page_number']}")

save_parsed('파싱_4_llamaparse.md',
            '\n'.join(f"<!-- {pg['page_number']}쪽 -->\n{pg['markdown']}" for pg in llama_pages))
save_parsed('파싱_4_llamaparse.json', json.dumps(result, ensure_ascii=False, indent=2))

**부르는 방식이 다릅니다.** Upstage 는 요청 한 번으로 결과까지 받았지만, LlamaParse 는 방금 본 **1) 파일 업로드 → 2) 작업 생성 → 3) 끝날 때까지 물어보기(폴링)** 세 단계입니다. 오래 걸리는 작업을 다루는 API 에서 흔한 방식이라 한 번은 겪어 둘 만합니다.

**돌려주는 형식도 다릅니다.** 이 티어의 LlamaParse 는 표를 **HTML `<table>`** 로 줍니다. 앞에서 쓴 "`|` 로 시작하는 줄을 세는" 방법을 그대로 쓰면 **표를 0개로 셉니다.** 그래서 맨 앞에서 만든 `table_rows`/`describe` 는 두 형식을 모두 읽어 **행과 칸의 목록**으로 바꾼 뒤에 셉니다.

이제 **셋을 원본과 대 봅니다.**

In [ ]:
local_first = pymupdf4llm.to_markdown(str(SAMPLE_PDF_PATH), pages=[0], show_progress=True)

print('----- 9쪽 표를 셋이 어떻게 잡았나 -----')
describe('pymupdf4llm', local_first)
describe('Upstage', up_first_table)
describe('LlamaParse', llama_first)

# 입력 PDF 도 지우지 않는다 -- 결과 파일들과 함께 열어 봐야 하고, 배포본과 정답본이 같은 output/ 폴더를
# 쓰기 때문에 한쪽이 지우면 다른 쪽이 읽으려던 파일이 사라진다. 덮어쓰기만 하면 그런 일이 없다.

9쪽 표의 참값은 **4열 · 데이터 22행**이고, 13~22행은 `표준(안)`+`비고` 가 병합돼 "개인정보 처리현황에 따라 작성" 한 문장이 걸쳐 있습니다.

| | 열 | 데이터 행 | 13~22행 중 끝 열이 있는 행 |
|---|---|---|---|
| **원본** | 4 | 22 | 문장이 10행에 걸침 |
| `pymupdf4llm` | **3** | **21** | 3 |
| Upstage | 4 | 22 | **10** |
| LlamaParse | 4 | 22 | **1** |

1~12행은 세 도구 모두 같게 나오므로, 갈리는 자리인 13~22행만 봅니다.

**로컬 도구만 열 하나를 잃고**(`표준(안)`+`비고` 를 합쳐서) 17·18행이 뭉쳐 21행이 됐습니다. 상용 둘은 열도 행도 원본과 같습니다. 32쪽 표 두 개는 셋 다 정확했습니다.

병합된 그 문장을 Upstage 는 **10행 전부에 채우고**, LlamaParse 는 **한 곳에만** 넣습니다.

- 화면으로 볼 때는 LlamaParse 가 원본 레이아웃에 더 충실합니다.
- 표를 잘라 색인해 **검색 근거**로 쓸 때는 각 행이 자기 값을 갖는 편이 낫습니다 — LlamaParse 결과에서는 13~22행 조각에 "이 항목은 어떻게 작성하나"가 **아예 없습니다.** 그러니 **"무엇이 더 정확한가"가 아니라 "내가 이걸 어디에 쓸 것인가"** 의 문제입니다.

LlamaParse 는 `cost_effective` 티어로 잰 값이라 티어를 올리면 달라질 수 있습니다.

### 결과물을 직접 열어 보세요

입력과 네 도구의 결과를 전부 `output/` 에 파일로 남겼습니다.

| 파일 | 무엇 |
|---|---|
| `파싱_0_입력_2쪽.pdf` | **넣은 입력** — 원본에서 9·32쪽만 잘라 낸 PDF |
| `파싱_1_get_text.txt` | PyMuPDF `get_text()` — 구조 없는 날 글자 |
| `파싱_2_to_markdown.md` | `pymupdf4llm` — 마크다운 구조 |
| `파싱_3_upstage.md` · `.json` | Upstage · `.json` 은 응답 원본이라 `elements` 배열을 볼 수 있습니다 |
| `파싱_4_llamaparse.md` · `.json` | LlamaParse |

**VS Code 에서 나란히 열어** 9쪽 표의 머리 줄부터 비교해 보세요.

### 표 말고 그림은 — 도식이 많은 문서로

논문·설계서에는 **도식(그림)** 이 더 많습니다. **Attention Is All You Need**(트랜스포머 논문)의 **3·4쪽**(Figure 1·2)으로 확인합니다. 볼 것은 셋입니다 — 그림 **파일**을 꺼내 주는가, 그림 **안에 그려진 글자**(`Add & Norm` 같은 도식 라벨)를 읽어 주는가, **캡션**을 어떻게 다루는가.

In [ ]:
import base64

PAPER_PDF_PATH = Path('data/원본/attention is all you need.pdf')
FIG_PAGES = [3, 4]           # Figure 1 · 2 가 있는 쪽(뷰어 번호)

# 앞에서 한 것과 같은 방식 -- 필요한 두 쪽만 잘라 낸다.
FIG_PDF_PATH = OUT_DIR / '그림_0_입력_2쪽.pdf'
paper = pymupdf.open(PAPER_PDF_PATH)
cut = pymupdf.open()
for page_no in FIG_PAGES:
    cut.insert_pdf(paper, from_page=page_no - 1, to_page=page_no - 1)
cut.save(FIG_PDF_PATH)
print(f'저장: {FIG_PDF_PATH} ({cut.page_count}쪽)')

# 도식 안에만 있는 라벨을 고른다 -- 'Multi-Head Attention' 처럼 본문에도 나오는 말을 쓰면
# 그림에서 읽었는지 본문에서 주웠는지 구분이 안 된다.
LABELS = ['Add & Norm', 'Softmax', 'MatMul']
def label_hits(text):
    """도식 안에 그려진 라벨이 텍스트로 잡혔는지 센다."""
    return sum(1 for w in LABELS if w in text)

# (1) 로컬 기본값 -- 그림을 아예 건드리지 않는다.
plain = pymupdf4llm.to_markdown(str(FIG_PDF_PATH), show_progress=True)
save_parsed('그림_1_to_markdown_기본.md', plain)
print(f'  기본값        : 그림 링크 {plain.count("![")}개 · 도식 라벨 {label_hits(plain)}/{len(LABELS)}개')

# (2) write_images=True -- 그림을 PNG 로 저장하고 마크다운에 링크를 넣는다.
PNG_DIR = OUT_DIR / '그림_png'
PNG_DIR.mkdir(exist_ok=True)
with_img = pymupdf4llm.to_markdown(str(FIG_PDF_PATH), write_images=True,
                                   image_path=str(PNG_DIR), show_progress=True)
save_parsed('그림_2_to_markdown_이미지.md', with_img)
n_png = len(list(PNG_DIR.glob('*.png')))
print(f'  write_images  : 그림 링크 {with_img.count("![")}개 · PNG {n_png}개 저장 · 도식 라벨 {label_hits(with_img)}/{len(LABELS)}개')

# (3) Upstage -- figure 원소를 잘라 base64 로 돌려받는다.
if not upstage_key:
    raise RuntimeError('UPSTAGE_API_KEY 가 필요합니다 -- https://console.upstage.ai 에서 발급해 .env 에 넣으세요.')

with open(FIG_PDF_PATH, 'rb') as fp:
    fig_res = requests.post(
        'https://api.upstage.ai/v1/document-digitization',
        headers={'Authorization': f'Bearer {upstage_key}'},
        files={'document': fp},
        data={'model': 'document-parse', 'output_formats': '["markdown"]',
              'coordinates': 'false', 'base64_encoding': '["figure"]'},
        timeout=180)
if fig_res.status_code != 200:
    raise RuntimeError(f'Upstage 요청 실패({fig_res.status_code}): {fig_res.text[:200]}')

fig_parsed = fig_res.json()
save_parsed('그림_3_upstage.md', fig_parsed['content']['markdown'])
save_parsed('그림_3_upstage.json', json.dumps(fig_parsed, ensure_ascii=False, indent=2))

figures = [e for e in fig_parsed['elements'] if e['category'] == 'figure']
captions = [e for e in fig_parsed['elements'] if e['category'] == 'caption']
hits = label_hits(fig_parsed['content']['markdown'])
print(f'  Upstage       : figure 원소 {len(figures)}개 · caption 원소 {len(captions)}개 · 도식 라벨 {hits}/{len(LABELS)}개')

# base64 로 온 그림을 PNG 로 풀어 저장한다 -- 열어 보면 도식이 그대로 잘려 있다.
for n, el in enumerate(figures, 1):
    blob = el.get('base64_encoding')
    if blob:
        out = OUT_DIR / f'그림_3_upstage_figure{n}.png'
        out.write_bytes(base64.b64decode(blob))
        print(f'    저장: {out} ({out.stat().st_size / 1024:.0f}KB)')

# (4) LlamaParse -- 그림 정보를 따로 목록으로 준다.
# 키가 없으면 멈춘다 -- 우회로를 두지 않는다.
if not llama_key:
    raise RuntimeError('LLAMA_CLOUD_API_KEY 가 필요합니다 -- https://cloud.llamaindex.ai 에서 발급해 .env 에 넣으세요.')

head = {'Authorization': f'Bearer {llama_key}'}
with open(FIG_PDF_PATH, 'rb') as fp:
    fid = requests.post(f'{LLAMA_BASE}/api/v1/beta/files', headers=head,
                        files={'file': (FIG_PDF_PATH.name, fp, 'application/pdf')},
                        data={'purpose': 'parse'}, timeout=180).json()['id']
jid = requests.post(f'{LLAMA_BASE}/api/v2/parse', headers=head,
                    json={'file_id': fid, 'tier': 'cost_effective', 'version': 'latest'},
                    timeout=180).json()['id']
fig_job = {}
for _ in range(20):
    # expand 에 적은 것만 채워져 온다 -- 그림 정보를 받으려면 이름을 함께 적어야 한다.
    # (expand=images 는 없는 값이라 400 이 온다. 맞는 이름은 images_content_metadata 다.)
    fig_job = requests.get(
        f'{LLAMA_BASE}/api/v2/parse/{jid}?expand=markdown,images_content_metadata',
        headers=head, timeout=180).json()
    if fig_job.get('job', {}).get('status') in ('COMPLETED', 'ERROR', 'CANCELED'):
        break
    time.sleep(3)

if fig_job.get('job', {}).get('status') != 'COMPLETED':
    raise RuntimeError(f"LlamaParse 작업이 끝나지 않았습니다(상태: {fig_job.get('job', {}).get('status')}).")

fig_pages = fig_job['markdown']['pages']
fig_md = '\n'.join(pg['markdown'] for pg in fig_pages)
save_parsed('그림_4_llamaparse.md', fig_md)
save_parsed('그림_4_llamaparse.json', json.dumps(fig_job, ensure_ascii=False, indent=2))
# 그림 목록은 쪽마다가 아니라 응답 맨 위에 있고, 개수는 total_count 에 담겨 온다.
img_meta = fig_job.get('images_content_metadata') or {}
n_img = img_meta.get('total_count', len(img_meta.get('images', [])))
print(f'  LlamaParse    : 그림 정보 {n_img}개 · 도식 라벨 {label_hits(fig_md)}/{len(LABELS)}개')

**그림 파일을 실제로 저장할 수 있는 것은 로컬과 Upstage 둘입니다.** 로컬은 `write_images=True` 면 PNG 로 저장하고 `![](경로)` 링크를 넣습니다(기본값이 꺼져 있을 뿐). Upstage 는 `base64_encoding` 옵션으로 그림 바이트를 응답에 함께 실어 줍니다. **LlamaParse 는 이 응답 안에 파일명·타입만 줍니다** — 실제 이미지를 받으려면 그 정보로 별도 다운로드 엔드포인트를 한 번 더 불러야 하고, 이 실습에서는 거기까지 가지 않습니다.

| 도구 | 그림 파일 | 그림 안 글자 | 캡션 |
|---|---|---|---|
| `pymupdf4llm` 기본값 | 안 가져옴 | **없음** | 캡션 텍스트만 |
| `pymupdf4llm` `write_images=True` | **PNG + 링크** | **없음** | 캡션 텍스트만 |
| Upstage | `figure` 원소 + **잘라낸 PNG** | **읽어 냄** | `caption` 원소로 분리 |
| LlamaParse | 그림 정보만(파일명·타입) — **파일은 별도 호출 필요** | **읽어 냄** | 본문에 이어 붙음 |

**갈리는 것은 그림 "안"의 글자입니다.** 로컬은 도식을 그림 파일로만 남기므로 그 내용은 **검색에 걸리지 않습니다** — "Add & Norm 층이 어디에 있나"에 답할 근거가 색인에 없습니다. **표에서 열 하나를 잃은 것과 같은 이야기입니다.**

`output/그림_3_upstage_figure1.png` 를 열어 보세요 — Figure 1 도식만 정확히 잘려 있습니다. `그림_png/` 에는 로컬 도구가 뽑은 PNG 가 들어 있습니다.

### 파싱 옵션 — 실무에서 손대는 것만

`to_markdown` 의 인자는 28개나 됩니다. 실제로 손대는 것은 몇 개뿐입니다.

| 옵션 | 기본값 | 무엇 |
|---|---|---|
| `pages` | `None` | 필요한 쪽만 읽는다(0부터) |
| `page_chunks` | `False` | 쪽 단위 목록 + `metadata['page_number']` |
| `write_images` | `False` | 그림을 파일로 저장하고 `![](경로)` 삽입 |
| `image_path` · `image_format` | `''` · `'png'` | 그림 저장 위치·형식 |
| `embed_images` | `False` | 그림을 base64 로 본문에 박는다(파일 대신) |
| `ignore_images` · `ignore_graphics` | `False` | 그림을 아예 건너뛴다(속도) |
| `table_strategy` | `'lines_strict'` | 표를 어떻게 찾을지 (`'lines'` · `'text'` 도 있음) |
| `margins` | `0` | 위아래 여백을 잘라 낸다 — **머리말·꼬리말 제거**에 쓴다 |
| `show_progress` | `False` | 진행 상황을 보여 준다. **오래 걸리는 문서에서 멈춘 게 아님**을 알 수 있다 |

전체 목록은 `help(pymupdf4llm.to_markdown)` 이나 공식 문서에서 봅니다.

상용 API 도 옵션이 있습니다. **Upstage** 는 `model` · `mode`(`standard`·`enhanced`·`auto`) · `output_formats` · `ocr`(`auto`·`force`) · `coordinates` · `chart_recognition` · `merge_multipage_tables` · `base64_encoding` · `words`. **LlamaParse** 는 `tier`(`fast`·`cost_effective`·`agentic`·`agentic_plus`) · `version` · `expand`.

> `expand` 에 **적지 않은 항목은 응답에 `None` 으로** 옵니다 — 없는 게 아니라 **요청하지 않은 것**입니다. 그림 정보를 받으려면 `expand=markdown,images_content_metadata` 처럼 이름을 함께 적어야 합니다(`expand=images` 는 없는 값이라 400 이 돌아옵니다).

**그럼 옵션을 바꾸면 아까 그 표도 고쳐질까요?** `table_strategy` 를 세 값으로 돌려 봅니다.

In [ ]:
# 표를 찾는 방식을 바꾸면 병합된 열이 되살아날까 -- 직접 확인한다(로컬 연산이라 비용이 없다).
def head_lines(text):
    """연속된 | 줄 덩어리마다 첫 줄(머리 줄)을 모은다."""
    heads, inside = [], False
    for line in text.splitlines():
        if line.strip().startswith('|'):
            if not inside:
                heads.append(line.strip())
            inside = True
        else:
            inside = False
    return heads

for strategy in ['lines_strict', 'lines', 'text']:
    text = pymupdf4llm.to_markdown(str(SAMPLE_PDF_PATH), table_strategy=strategy, show_progress=True)
    heads = head_lines(text)
    # 양끝 | 를 하나씩만 떼고 센다(앞에서 본 그 규칙).
    cols = [len(h[1:-1].split('|')) if h.endswith('|') else len(h[1:].split('|')) for h in heads]
    print(f"table_strategy={strategy:14} 표 {len(heads)}개 · 열 {cols}")

print(f'\n첫 표 머리 줄: {head_lines(text)[0][:60]}')

**세 방식 모두 같습니다.** 옵션을 바꿔서 되는 문제였다면 도구를 바꿀 이유가 없습니다. 그 표의 병합된 열은 **이 도구의 한계**이지 설정 실수가 아닙니다.

### 뽑는 것까지가 끝이 아니다 — 무엇을 남기고 무엇을 버릴지

`to_markdown` 은 **그림 안에 그려진 글자까지** 읽어 줍니다. 고마운 기능이지만, 이 안내서들은 본문 문단을 **그림으로도 한 번 더** 싣습니다. 그대로 두면 같은 문단이 두 벌씩 색인돼 검색 결과 자리를 낭비합니다. 다행히 이 도구는 그림에서 온 글자를 **표시로 감싸** 알려 줍니다 — 아래 셀에서 표시 바로 앞 본문과 그림 글자가 **같은 문장**임을 확인하세요.

그래서 이 강의의 코퍼스를 만들 때는 **그 표시로 감싼 덩어리와, 쪽마다 반복되는 머리글**(`Ⅲ. … <u>15</u>` 처럼 장 제목과 인쇄 쪽번호가 붙은 첫 줄)을 걸러 냈습니다. 그 규칙은 코퍼스를 만드는 빌드 스크립트에 들어 있습니다(강사용 저장소).

여기서 얻을 교훈은 **정규식으로 본문을 손질하라**가 아닙니다. 정반대입니다. 걸러 낸 근거가 **도구가 스스로 붙여 준 표시**(`<!-- ... -->`)와 **매 쪽 똑같이 반복되는 줄**이었지, 글의 내용을 읽고 판단한 것이 아닙니다. 그래서 문서가 바뀌어도 잘 버팁니다. 반대로 본문 문장을 눈으로 훑어 가며 정규식을 깎기 시작하면, 문서가 조금만 달라져도 규칙이 깨지고 정작 필요한 내용까지 지워집니다.

**무엇을 남기고 무엇을 버릴지 정하는 것까지가 파싱입니다.** 다만 그 판단의 근거는 **도구가 준 표시와 문서의 반복 구조**여야 합니다.

In [ ]:
# 도구가 그림에서 뽑은 글자에 붙여 준 표시를 센다.
MARK_START = '<!-- Start of picture text -->'

marked = [p['metadata']['page_number'] for p in pages if MARK_START in p['text']]
print(f"그림 글자가 들어 있는 쪽: {len(marked)}쪽 -> {marked[:10]} ...")

# 27쪽에서 그 표시가 감싼 대목을, 바로 앞 본문과 함께 꺼내 본다 -- 겹치는지 눈으로 보려면 둘 다 필요하다.
page27 = [p for p in pages if p['metadata']['page_number'] == 27][0]['text']
start = page27.index(MARK_START)
end = page27.index('<!-- End of picture text -->') + len('<!-- End of picture text -->')

print('\n----- 표시 바로 앞의 본문 -----')
print(page27[max(0, start - 200):start].strip())
print('\n----- 그림에서 온 글자 -----')
print(page27[start:end][:300])

### 🖐️ 함께 따라하기

이번엔 **다른 PDF** 를 읽어 봅니다. 같은 폴더의 **`개인정보_처리방침_표준안.pdf`** 입니다.

1. 경로를 `MY_PDF_PATH` 에 담고 `pymupdf4llm.to_markdown(..., page_chunks=True, show_progress=True)` 로 쪽 목록 `pages_pp` 를 만든다.
2. 전체 쪽 수를 출력한다.
3. `metadata['page_number']` 가 **9** 인 쪽을 찾아, 위에서 만든 `count_marks` 로 제목 줄 수와 표 줄 수를 세어 출력한다.
4. 같은 쪽을 `pymupdf.open(...)` 으로 열어 `get_text()` 로도 뽑아, 두 결과의 **글자 수**를 나란히 출력한다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. `to_markdown(경로, page_chunks=True)` 가 돌려준 쪽 하나에서 **쪽 번호**를 꺼내는 표현은 무엇인가요?
2. 줄글만 빽빽한 계약서 5,000쪽을 빠르게 훑어 검색 색인을 만들려 합니다. 어느 도구가 나을까요? 그 이유는?
3. PDF 뷰어에서 본 **31쪽**만 파싱하려면 `pages` 에 무엇을 넣어야 하고, 돌아온 결과의 `metadata['page_number']` 는 얼마일까요?

<details><summary>정답 보기</summary>

1. `쪽['metadata']['page_number']` — `쪽['page']` 가 아니다.
2. `get_text`. 표·제목처럼 살려야 할 구조가 없는 문서라 마크다운으로 얻을 이득이 없는데, 실측에서 본 것처럼 `to_markdown` 은 시간이 훨씬 더 든다.
3. `pages=[30]`(0부터 세므로 하나 뺀다). 돌아온 `page_number` 는 **31** 이고, 출처로 적을 번호도 이쪽이다.

</details>

## 2. 왜 자르나 — 청킹

앞 절에서 만든 쪽 하나가 곧 '문서 하나'가 됩니다. 그런데 **임베딩 모델은 아무리 긴 글이라도 앞에서 정해진 길이까지만 읽습니다.** 그 길이를 넘긴 뒷부분은 벡터에 조금 섞이는 것이 아니라 **아예 들어가지 않습니다.**

여기가 헷갈리기 쉬운 대목입니다. 긴 편지를 짧은 봉투에 넣으면 뒷장이 잘려 나가듯, 잘린 뒷부분은 흐려지는 게 아니라 **처음부터 없던 것**이 됩니다. **청킹**은 편지를 봉투에 맞게 여러 통으로 나눠 부치는 일입니다.

말로 믿지 말고 **모델에게 직접 물어봅시다.** 우리가 쓰는 모델이 몇 개까지 읽는지, 그리고 쪽 하나가 그 몇 배인지 나란히 찍어 보면 됩니다.

그 전에 오늘 쓸 코퍼스부터 봅니다. 앞 절에서 한 파싱을 두 PDF 에 그대로 돌려 **쪽 하나 = 행 하나**로 저장해 둔 것이 `guide_docs.csv` 입니다. 방금 이야기한 **그림 글자와 반복 머리글은 걸러 낸 뒤**라서, 같은 쪽이라도 글자 수가 `to_markdown` 원본보다 짧습니다.

In [ ]:
guide = pd.read_csv('data/guide_docs.csv')

print("행·열 크기:", guide.shape)
print("\n[열 정보]"); guide.info()
print("\n[문서별 쪽 수]"); print(guide["문서"].value_counts())
print("\n[앞 3행]"); display(guide.head(3))

In [ ]:
# 쪽 하나가 얼마나 긴가 -- 청킹이 필요한지 판단하는 첫 숫자다.
lengths = guide['본문'].str.len()

print(f"본문 길이  평균 {lengths.mean():,.0f}자 · 최소 {lengths.min():,}자 · 최대 {lengths.max():,}자")

# 빈 줄로 나뉜 문단이 실제로 있는지도 본다 -- 문단 단위로 자를 수 있는지가 여기서 갈린다.
paras = guide['본문'].apply(lambda t: len([p for p in t.split('\n\n') if p.strip()]))
print(f"쪽마다 문단 수  중앙값 {paras.median():.0f}개 · 최소 {paras.min()}개 · 최대 {paras.max()}개")

쪽 하나가 천 자를 훌쩍 넘습니다. 그럼 **모델은 그중 얼마나 읽을까요.** 임베딩 모델은 글을 글자가 아니라 **토큰**(모델이 세는 낱말 조각)으로 셉니다. 읽을 수 있는 토큰 수의 상한이 `max_seq_length` 입니다.

In [ ]:
# 모델이 한 번에 읽는 토큰 수의 상한. 이 값이 청킹의 첫째 이유다.
LIMIT = embed_model.max_seq_length
print(f'모델이 읽는 상한: {LIMIT} 토큰')

# 쪽 하나가 몇 토큰인지 세어 본다 -- 토큰 세는 도구는 모델이 함께 갖고 있다.
# verbose=False 는 '상한을 넘었다' 는 경고를 끈다. 우리는 세기만 하고 모델에 넣지 않으니 문제가 없다.
tokenizer = embed_model.tokenizer
token_counts = guide['본문'].apply(lambda text: len(tokenizer.encode(text, verbose=False)))

over = (token_counts > LIMIT).sum()
print(f'쪽 토큰 수  중앙값 {token_counts.median():.0f} · 최대 {token_counts.max()}')
print(f'상한을 넘는 쪽: {len(guide)}개 중 {over}개')

**모든 쪽이** 상한을 넘습니다. 넘긴 뒷부분은 **읽히지 않습니다.**

이게 정말인지 확인할 방법이 있습니다. 쪽을 통째로 임베딩한 벡터와, **그 쪽의 앞부분만** 임베딩한 벡터를 견줘 보면 됩니다. 뒷부분이 조금이라도 반영됐다면 두 벡터는 달라야 합니다.

27쪽(`ai27`)으로 해 봅니다. '서비스형 LLM' 이야기와 '개인용 vs 기업용 API 라이선스 비교표'가 함께 있는 쪽이라, **라이선스만 묻는 질문**을 던지면 앞뒤 어디가 필요한지도 함께 보입니다.

In [ ]:
sample = guide[guide['id'] == 'ai27'].iloc[0]
sample_text = sample['본문']
question = '기업용 API 라이선스는 개인용과 무엇이 다른가요?'

# 질문과 '쪽 통째' 를 각각 벡터로 만든다(길이를 1로 맞췄으니 내적이 곧 코사인 유사도다).
q_vec = embed_model.encode([question], normalize_embeddings=True)[0]
whole_vec = embed_model.encode([sample_text], normalize_embeddings=True)[0]

# 같은 쪽의 '앞 400자만' 도 벡터로 만든다 -- 뒷부분이 반영됐다면 두 벡터는 달라야 한다.
head_vec = embed_model.encode([sample_text[:400]], normalize_embeddings=True)[0]

print(f"27쪽: {len(sample_text):,}자 = {len(tokenizer.encode(sample_text, verbose=False))}토큰  (상한 {LIMIT})")
print(f"앞 400자      = {len(tokenizer.encode(sample_text[:400], verbose=False))}토큰")

print(f"\n질문 <-> 쪽 통째   유사도 {float(q_vec @ whole_vec):.6f}")
print(f"질문 <-> 앞 400자  유사도 {float(q_vec @ head_vec):.6f}")
print(f"\n쪽 통째 <-> 앞 400자  코사인 {float(whole_vec @ head_vec):.6f}")

**두 벡터가 같습니다.** 1,348자짜리 쪽을 통째로 넘겨 만든 벡터가, 앞 400자만 넘겨 만든 벡터와 소수점 아래까지 똑같습니다. 뒤의 900자는 **희미하게 반영된 것이 아니라 아예 읽히지 않았습니다.**

> 그러니 "쪽 통째" 라는 말은 사실 **"그 쪽의 앞부분"** 이라는 뜻입니다. 자르지 않으면 뒷부분은 검색으로 **영영 찾을 수 없습니다** — 색인에 아예 없기 때문입니다.

이제 쪽을 잘라, 라이선스 이야기가 든 조각과 견줘 봅니다.

**세 가지 청킹 전략**을 만듭니다.

| 전략 | 자르는 기준 | 노리는 것 |
|---|---|---|
| `chunk_fixed(text, size)` | 글자 수로 딱딱 끊는다 | 가장 단순하고 빠르다 |
| `chunk_overlap(text, size, overlap)` | 고정 크기로 끊되 **앞 조각의 끝을 겹쳐** 갖는다 | 경계에 걸린 문장이 양쪽에 다 남는다 |
| `chunk_paragraph(text, size)` | **빈 줄로 나뉜 문단**을 모으다가 크기를 넘기 직전에 끊는다 | 문서가 원래 갖고 있던 경계를 존중한다 |

In [ ]:
# 세 가지 청킹 전략을 함수로 만든다. 셋 다 원문을 받아 '조각 목록' 을 돌려준다.
def chunk_fixed(text, size):
    """고정 크기(글자 수)로 자른다."""
    return [text[i:i + size] for i in range(0, len(text), size)]

def chunk_overlap(text, size, overlap):
    """앞 청크의 끝 일부를 다음 청크가 겹쳐 갖도록 자른다."""
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)]

def chunk_paragraph(text, size):
    """빈 줄로 나뉜 문단을 순서대로 모아 size 근처에서 끊는다.

    한 문단 자체는 절대 쪼개지 않는다 -- 문단이 size 보다 길면 그 조각 하나가
    size 를 그대로 넘어선 채로 들어간다."""
    chunks, cur = [], ''
    for para in [p.strip() for p in text.split('\n\n') if p.strip()]:
        if cur and len(cur) + len(para) > size:
            chunks.append(cur)
            cur = para
        else:
            cur = f'{cur}\n{para}' if cur else para
    if cur:
        chunks.append(cur)
    return chunks

print("세 전략 준비 완료")

같은 쪽을 셋으로 잘라 **몇 조각이 나오고 각 조각이 몇 자인지** 먼저 봅니다.

In [ ]:
SIZE = 400        # 조각 하나의 목표 크기(글자 수)

# 같은 쪽(sample_text)을 세 전략으로 각각 잘라 둔다.
strategies = {
    '고정': chunk_fixed(sample_text, SIZE),
    '겹침': chunk_overlap(sample_text, SIZE, 80),
    '문단': chunk_paragraph(sample_text, SIZE),
}

# 전략마다 조각이 몇 개 나왔고 각 조각이 몇 자인지 본다.
for name, chunks in strategies.items():
    sizes = [len(c) for c in chunks]
    print(f'{name}: {len(chunks)}조각 · 길이 {sizes}')

이제 **조각의 끝**을 봅니다. 자른 자리가 말이 되는 자리인지가 여기서 드러납니다.

In [ ]:
# 조각들이 '어디서 끝났는지' 를 본다 -- 낱말 한가운데면 뜻이 반토막 난 것이다.
for name in ['고정', '문단']:
    print(f'[{name}]')
    for i, chunk in enumerate(strategies[name]):
        print(f'   {i}번 조각 끝 ...{chunk[-30:]!r}')

고정 크기는 글자 수만 세기 때문에 **낱말 한가운데**에서도 그냥 끊습니다. 문단 방식은 **원문이 갖고 있던 경계(빈 줄)** 에서만 끊습니다 — 낱말이 잘리지는 않지만, 그 경계가 꼭 문장 끝인 것은 아닙니다. 위 출력에서 보듯 **표의 한 행**이나 **목록 항목**에서 끝나기도 합니다. 원문의 구조를 그대로 따라가는 것이지, 문장을 알아보고 자르는 것이 아니기 때문입니다.

마지막으로 **검색이 실제로 나아지는지** 재 봅니다. 세 전략의 조각을 각각 임베딩해, 아까 그 질문과 **가장 가까운 조각의 유사도**를 봅니다. 앞에서 잰 '쪽 통째' 값과 견주면 됩니다.

In [ ]:
print(f'질문: {question}')
print(f'쪽 통째        유사도 {float(q_vec @ whole_vec):.4f}')

# 전략마다 조각을 임베딩해, 질문과 가장 가까운 조각의 유사도를 뽑는다.
for name, chunks in strategies.items():
    chunk_vecs = embed_model.encode(chunks, normalize_embeddings=True)
    sims = chunk_vecs @ q_vec
    best = int(np.argmax(sims))
    print(f'{name} 청킹 최고   유사도 {sims[best]:.4f}  (조각 #{best}, {len(chunks[best])}자)')

세 전략 모두 **쪽 통째보다 높습니다** — 이것이 자르는 이유입니다. 라이선스 비교표는 그 쪽의 **뒤쪽**에 있어서 통째로 넘겼을 때는 모델이 읽지도 못했는데, 잘라 놓으니 그 대목이 **제 몫의 벡터**를 갖게 된 것입니다.

**그럼 셋 중 무엇을 고르나.** 유사도 숫자만 보면 고정과 문단이 거의 붙어 있고 겹침이 조금 낮습니다. 숫자 차이가 이렇게 작을 때는 **다른 근거로 판단해야 합니다.**

- 이 코퍼스는 파싱 결과에 **빈 줄로 나뉜 덩어리가 이미 또렷하게** 들어 있습니다(위에서 쪽마다 세어 봤습니다). 그 덩어리는 문단일 수도, 목록 항목이나 표의 한 행일 수도 있지만 **어느 쪽이든 원문이 스스로 그어 둔 선**입니다.
- 문단 방식은 그 선에서만 끊으므로 **낱말이 잘리지 않습니다.** 고정 방식이 `▲입력데이` 처럼 글자 한가운데를 가르는 것과 대비됩니다.
- 겹침 방식은 유사도가 특별히 좋아지진 않으면서 조각 수가 늘어 **저장·검색 비용이 커집니다.** 대신 경계에 걸린 문장을 놓치지 않는 보험이라, 원문에 그어진 선이 없는 문서에서 값을 합니다.

그래서 **이 문서에는 문단 방식**을 씁니다. 그런 선이 없는 자료(자막·채팅 로그·OCR 결과)였다면 겹침 방식이 답이었을 것입니다. **도구를 먼저 정하고 데이터를 맞추는 게 아니라, 데이터를 보고 도구를 고르는 순서**입니다.

> 한 가지 정직하게 덧붙이면, 문단 방식은 **덩어리 하나가 `size` 보다 길면 그것을 쪼개지 않습니다.** 그래서 가끔 `size` 를 넘는 조각이 나옵니다. 조각 길이의 상한이 꼭 필요한 상황이라면 이 점을 알고 써야 합니다.

### 그럼 조각을 상한(128토큰)에 딱 맞추면 더 좋아지나

당연히 그럴 것 같지만, **재 보면 그렇지 않습니다.** 이 코퍼스에서 `SIZE` 를 400 → 250자로 줄이면 대부분의 조각이 상한 안에 들어오는데도, 15개 질문으로 잰 검색 성적은 **오히려 떨어집니다.**

잘려서 잃는 것과, 조각이 짧아져 **앞뒤 문맥을 잃는 것**이 서로 상쇄되기 때문입니다. 조각이 너무 짧으면 그 조각만 봐서는 무슨 이야기인지 알 수 없어, 질문과 맞춰 보기가 오히려 어려워집니다.

그래서 이 강의는 **`SIZE = 400` 을 그대로 씁니다** — 상한을 넘는 조각이 남는다는 것을 알면서도요. **청크 크기·겹침 값을 하나씩 바꿔 가며 실제로 재 보는 일은 다음 시간**에 제대로 합니다. 지금 기억할 것은 이것입니다: **"상한이 있다"는 사실과 "상한에 맞추면 좋아진다"는 기대는 다른 이야기이고, 후자는 재 봐야 안다.**

### 실무에서는 보통 어떻게 자를까요?

실무의 출발점은 셋 중 **하나를 고르는 것이 아니라 둘을 겹쳐 쓰는 것**입니다. 한 문장으로 하면 이렇습니다.

> **자연 경계를 우선 시도하되, 그래도 길면 글자 수로 자르고, 조각끼리 조금 겹치게 한다.**

셋 다 우리가 이 절에서 이미 본 이유들입니다.

- **자연 경계 우선** — 원문이 그어 둔 선에서 끊으면 낱말이 잘리지 않습니다. 고정 방식이 `▲입력데이` 처럼 글자 한가운데를 가르던 것과 대비됩니다.
- **그래도 글자 수 상한이 필요한 이유** — 문단 방식은 **덩어리 하나가 `size` 보다 길면 쪼개지 않기** 때문입니다. 실제로 이 코퍼스를 `chunk_paragraph(400)` 으로 자르면 **245조각 중 16개가 400자를 넘고, 가장 긴 것은 1,287자**입니다. 빈 줄이 없는 표가 통째로 한 조각이 되기 때문입니다. 그런 조각은 앞부분만 읽히므로, 자연 경계로 자른 뒤에도 **너무 길면 한 번 더 잘라 주어야** 합니다.
- **겹침이 필요한 이유** — 경계에 걸린 문장을 양쪽 조각이 모두 갖게 해 놓치지 않기 위해서입니다. 겹치는 양은 보통 조각 길이의 **10~20%** 를 씁니다. 이 절에서 써 본 `400 · 80` 이 정확히 **20%** 입니다.

널리 쓰이는 문서 분할 도구들도 대개 이 방식을 기본값으로 내놓습니다. **오늘 만든 세 함수는 따로 노는 셋이 아니라, 그 기본값을 이루는 조각들**입니다.

다만 여기까지가 **"대체로 이렇게 시작한다"** 입니다. 어떤 값이 이 문서에 맞는지는 앞에서 본 것처럼 **재 봐야 압니다** — 겹침이 실제로 검색을 얼마나 끌어올리는지는 다음 시간에 지표로 확인합니다.

<img src="images/청킹_세전략.png" width="900">

### 🖐️ 함께 따라하기

**다른 쪽·다른 질문**으로 같은 비교를 해 봅니다. 대상은 `guide` 에서 `id` 가 **`pp9`** 인 행(처리방침 표준안 9쪽), 질문은 **"개인정보 처리방침에는 어떤 항목을 적어야 하나요?"** 입니다.

1. `pp9` 행의 `본문` 을 `my_text` 에 담고, 질문을 `my_question` 에 담는다.
2. 질문과 `my_text` 를 각각 `embed_model.encode([...], normalize_embeddings=True)[0]` 로 벡터로 만들어, **쪽 통째 유사도**를 출력한다.
3. `chunk_fixed(my_text, 400)` 과 `chunk_paragraph(my_text, 400)` 으로 각각 자르고, 조각들을 임베딩해 **최고 유사도**를 출력한다.
4. 세 값을 나란히 보고, 이 쪽에서는 어느 방식이 질문에 가장 가까운 조각을 만들었는지 확인한다.

> **쪽 통째 값과 고정 청킹 값이 똑같이 나올 수 있습니다.** 오류가 아닙니다 — 앞에서 본 그 이유입니다. 고정 청킹의 첫 조각은 그 쪽의 앞부분이고, 모델은 통째로 넘겨도 **어차피 앞부분까지만** 읽습니다. 그래서 두 벡터가 같아집니다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. 긴 쪽을 통째로 임베딩하면 뒷부분은 벡터에 어떻게 반영되나요?
2. 원문에 빈 줄 경계가 전혀 없는 자막 파일을 자르려 합니다. 세 전략 중 무엇이 알맞고, 그 이유는?
3. 조각을 모델의 상한(128토큰)에 딱 맞게 줄이면 검색이 좋아질까요?

<details><summary>정답 보기</summary>

1. **전혀 반영되지 않는다.** 모델은 `max_seq_length` 토큰까지만 읽고 나머지는 버린다. 흐릿하게 섞이는 것이 아니라 **없는 셈**이 되어, 자르지 않으면 그 대목은 검색으로 찾을 수 없다.
2. **겹침(`chunk_overlap`)**. 원문이 그어 둔 선이 없으니 문단 방식은 통째와 다를 바 없고, 고정 크기는 말 한가운데를 끊는다. 겹치게 자르면 경계에 걸린 문장이 양쪽 조각에 모두 남는다.
3. **재 봐야 안다.** 이 코퍼스에서는 400자에서 250자로 줄이자 성적이 오히려 떨어졌다 — 잘려서 잃는 것과 문맥이 짧아져 잃는 것이 상쇄되기 때문이다. 상한이 있다는 사실과 상한에 맞추면 좋아진다는 기대는 다른 이야기다.

</details>

## 3. 색인 — 한 번 만들어 두고 다시 쓴다

앞 절에서 자르는 방법을 정했으니, 이제 코퍼스 전체를 잘라 **한 번 만들어 두고 계속 쓸 색인**으로 만듭니다. 질문이 들어올 때마다 문서를 전부 임베딩할 수는 없습니다. 문서 쪽에서 할 일(자르기·임베딩·저장)은 **미리 한 번** 해 두고, 질문이 올 때는 질문만 임베딩해 꺼내 씁니다.

지난 시간엔 `EphemeralClient`(메모리)를 썼습니다. 오늘은 **`PersistentClient`** 를 씁니다 — 색인이 **디스크에 남아** 커널을 새로 켜도 다시 임베딩하지 않습니다.

메타데이터에는 네 가지를 싣습니다. 뒤 절에서 **필터**와 **출처 표기**에 그대로 쓰이는 값들입니다.

| 메타 키 | 값 | 어디에 쓰나 |
|---|---|---|
| `doc_id` | 원래 쪽의 id(`ai27` 등) | 같은 쪽의 다른 조각을 찾을 때 |
| `문서` | `생성형AI 안내서` / `처리방침 표준안` | 검색 필터 · 출처 표기 |
| `발간` | 2025 / 2026 | 검색 필터 |
| `쪽` | PDF 쪽 번호 | 출처 표기(뷰어에서 그 쪽을 연다) |

먼저 코퍼스 전체를 문단 방식으로 자릅니다.

In [ ]:
chunk_ids, chunk_texts, chunk_metas = [], [], []

for _, row in guide.iterrows():
    for i, chunk in enumerate(chunk_paragraph(row['본문'], SIZE)):
        # 조각 id 는 '쪽id-순번' 으로 짓는다 -> 나중에 앞뒤 조각을 번호로 찾을 수 있다.
        chunk_ids.append(f"{row['id']}-{i}")
        chunk_texts.append(chunk)
        chunk_metas.append({'doc_id': row['id'], '문서': row['문서'],
                            '발간': int(row['발간']), '쪽': int(row['쪽'])})

chunk_lengths = [len(c) for c in chunk_texts]
print(f"쪽 {len(guide)}개 -> 청크 {len(chunk_texts)}개")
print(f"청크 길이 평균 {np.mean(chunk_lengths):.0f}자 · 최대 {max(chunk_lengths):,}자")
print("\n첫 청크 메타:", chunk_metas[0])

이제 색인을 만듭니다. **한 번 만든 색인을 다시 쓰려면 '내용이 그대로인지' 확인할 방법**이 필요합니다. 청크 수만 비교하면 위험합니다 — 개수가 같아도 내용이 바뀌었을 수 있고, 특히 **메타데이터에 열이 하나 늘어난 경우** 낡은 색인에는 그 열이 없어 나중에 `KeyError` 가 납니다.

그래서 ids·texts·**metadatas 까지 전부** 해시로 요약한 **지문(fingerprint)** 을 색인에 함께 저장해 두고, 다음에 열 때 지문이 같을 때만 재사용합니다.

In [ ]:
import hashlib
from pathlib import Path

import chromadb

# output/ 는 실행 산출물 폴더다(저장소에 올라가지 않는다).
CHROMA_DIR = Path('output' if Path('data').exists() else '../output') / 'chroma'
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

def index_fingerprint(ids, texts, metadatas):
    """색인에 담긴 내용을 한 줄로 요약한 지문. 무엇 하나라도 바뀌면 값이 달라진다."""
    parts = ['\n'.join(ids), '\n'.join(texts),
             '\n'.join(repr(sorted(m.items())) for m in metadatas)]
    return hashlib.sha256('\x00'.join(parts).encode()).hexdigest()[:16]

def make_index(ids, texts, metadatas, name):
    """청크를 임베딩해 컬렉션으로 만든다. 같은 내용으로 이미 만들어 뒀으면 그대로 다시 쓴다."""
    want = index_fingerprint(ids, texts, metadatas)

    got = chroma.get_or_create_collection(name, metadata={'hnsw:space': 'cosine', 'fp': want})
    if got.count() == len(ids) and (got.metadata or {}).get('fp') == want:
        print(f'{name}: 만들어 둔 색인을 그대로 씁니다 (청크 {got.count()}개)')
        return got

    # 개수나 지문이 다르면 문서가 바뀐 것이다 -- 낡은 색인을 지우고 새로 만든다.
    chroma.delete_collection(name)
    col = chroma.create_collection(name, metadata={'hnsw:space': 'cosine', 'fp': want})

    emb = embed_model.encode(texts, normalize_embeddings=True)
    col.add(ids=ids, embeddings=emb.tolist(), documents=texts, metadatas=metadatas)

    print(f'{name}: 색인을 새로 만들었습니다 (청크 {col.count()}개)')
    return col

# 컬렉션 이름은 영문·숫자 3자 이상이어야 한다(ChromaDB 규칙 -- 한글 이름은 거부된다).
collection = make_index(chunk_ids, chunk_texts, chunk_metas, 'guide_chunks')

> 위 셀을 **한 번 더 실행해 보세요.** "만들어 둔 색인을 그대로 씁니다" 로 바뀌고 즉시 끝납니다. 임베딩을 다시 하지 않기 때문입니다.

만들어진 색인 안은 **`get`** 으로 들여다봅니다. `query` 가 질문과 가까운 것을 찾아 주는 검색이라면, `get` 은 **조건에 맞는 항목을 그냥 꺼내 오는** 기능입니다.

In [ ]:
# query 와 달리 get 은 질문 없이 꺼낸다. include 로 무엇을 함께 받을지 고른다.
peek = collection.get(limit=2, include=['metadatas', 'documents'])

print('꺼낸 id:', peek['ids'])
for meta, text in zip(peek['metadatas'], peek['documents']):
    print(f"  {meta} | {text[:40]!r}")

지문이 정말 제 몫을 하는지도 **일부러 깨뜨려** 확인해 봅시다. 청크 본문은 그대로 두고 **메타데이터에 키를 하나만 더해** 지문이 달라지는지 봅니다. 진짜 색인은 건드리지 않도록 **다른 이름**으로 만듭니다.

본문이 한 글자도 안 바뀌었는데 지문이 달라지는지 보세요. 청크 **수**만 비교했다면 이 변화를 놓치고 낡은 색인을 그대로 썼을 것이고, 나중에 `meta['출처파일']` 을 읽는 순간 `KeyError` 가 났을 것입니다.

In [ ]:
# 본문·id 는 그대로, 메타에만 키를 하나 더한다.
extra_metas = [dict(meta, 출처파일='guide_docs.csv') for meta in chunk_metas]

before = index_fingerprint(chunk_ids, chunk_texts, chunk_metas)
after = index_fingerprint(chunk_ids, chunk_texts, extra_metas)
print(f'메타 키 추가 전 지문: {before}')
print(f'메타 키 추가 후 지문: {after}')
print(f'같은가: {before == after}')

# 지문이 다르니 make_index 는 낡은 색인을 버리고 새로 만든다.
# 진짜 색인 guide_chunks 를 건드리지 않도록 이름을 따로 준다. 확인용이라 뒤에서 쓰지 않는다.
# 다 쓴 뒤에도 지우지 않는다 -- output/ 은 배포본과 정답본이 함께 쓰는 폴더라, 한쪽이 지우면
# 다른 쪽이 쓰던 것이 사라진다. 같은 이름으로 덮어쓰기만 하면 그런 일이 없다.
demo_col = make_index(chunk_ids, chunk_texts, extra_metas, 'guide_demo')

### 🖐️ 함께 따라하기

이번엔 **조건을 걸어** 색인 안을 들여다봅니다. `get` 에 `where` 를 주면 그 조건에 맞는 항목만 나옵니다.

1. `collection.count()` 로 전체 청크 수를 출력한다.
2. `collection.get(where={'문서': '처리방침 표준안'}, include=['metadatas'])` 로 그 문서의 청크만 꺼내 **개수**를 출력한다.
3. 꺼낸 메타에서 `쪽` 값을 모아 **가장 작은 쪽과 가장 큰 쪽**을 출력한다.
4. 같은 방법으로 `{'발간': 2025}` 인 청크 수도 출력해, 2번 결과와 합이 전체와 맞는지 확인한다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. 지문(fingerprint)을 **청크 본문만**이 아니라 **메타데이터까지** 넣어 계산하는 이유는 무엇인가요?
2. `EphemeralClient` 대신 `PersistentClient` 를 쓰면 무엇이 달라지나요?

<details><summary>정답 보기</summary>

1. 본문이 그대로여도 메타에 열이 늘거나 값이 바뀌면 낡은 색인에는 그 정보가 없다. 지문이 본문만 보면 그 변화를 놓쳐 색인을 재사용하고, 나중에 그 메타 키를 읽을 때 에러가 난다.
2. 색인이 **디스크에 남는다**. 커널을 새로 켜도 다시 임베딩하지 않고 그대로 이어 쓸 수 있다.

</details>

## 4. 검색 — 질문으로 청크 찾기

색인을 만들었으니 이제 꺼내 씁니다. 다만 **본문만 꺼내면 절반짜리**입니다. 뒤에서 답에 출처를 붙이려면 **어느 문서 몇 쪽에서 왔는지**를 함께 들고 와야 합니다.

`collection.query(...)` 는 결과를 몇 개의 목록으로 나눠 돌려줍니다. 질문을 여러 개 넣을 수 있는 구조라 **한 겹 더 감싸여** 있어서, 질문이 하나면 `[0]` 으로 벗겨야 합니다.

| 꺼내는 것 | 표현 | 내용 |
|---|---|---|
| 본문 | `res['documents'][0]` | 청크 글 |
| 메타 | `res['metadatas'][0]` | 우리가 실어 둔 `문서`·`쪽` 등 |
| 거리 | `res['distances'][0]` | 질문과 얼마나 먼가 |

**거리는 유사도의 반대입니다.** 컬렉션을 `hnsw:space='cosine'` 으로 만들었으므로 `거리 = 1 - 코사인 유사도` 입니다. **0에 가까울수록 가깝고**, 1에 가까우면 별 관계가 없습니다.

In [ ]:
def search(query, k=3, where=None):
    """질문과 가까운 청크 k개를 (조각 id, 본문, 메타, 거리) 로 돌려준다."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    res = collection.query(query_embeddings=q_emb.tolist(), n_results=k, where=where)
    # 질문 여러 개를 받을 수 있는 구조라 한 겹 감싸여 있다 -> [0] 으로 벗긴다.
    # id 도 함께 들고 온다 -- 뒤에서 '앞뒤 조각' 을 찾을 때 쓴다.
    return list(zip(res['ids'][0], res['documents'][0], res['metadatas'][0], res['distances'][0]))

def show(hits):
    """검색 결과를 문서명·쪽·거리와 함께 보여 준다."""
    for rank, (_, text, meta, dist) in enumerate(hits, 1):
        head = text.replace('\n', ' ')[:70]
        print(f"{rank}위 | 거리 {dist:.3f} | {meta['문서']} {meta['쪽']}쪽 | {head}...")

query = 'AI 학습에 쓸 데이터를 가명처리할 때 무엇을 확인해야 하나요'
print(f'질문: {query}\n')
show(search(query))

**거리값을 읽는 법.** 절대 기준이 있는 값은 아니지만, 이 코퍼스에서는 대략 이렇게 읽으면 됩니다.

- **0.2~0.3** — 질문이 가리키는 대목을 제대로 찾았다.
- **0.4 언저리** — 주제는 비슷하지만 정확히 그 이야기는 아닐 수 있다.
- **0.7 이상** — 자료에 그 내용이 아예 없다는 신호에 가깝다.
- **1위와 2위의 거리가 크게 벌어진다** — 답이 한 군데 뚜렷이 있다는 신호.
- **모두 고만고만하게 멀다** — 자료에 그 내용이 아예 없을 가능성이 크다.

마지막 경우를 직접 확인해 봅시다. **자료에 없는 것**을 물어보면 거리가 어떻게 나올까요.

In [ ]:
# 이 안내서에 있을 리 없는 질문 -- 거리가 어떻게 나오는지 본다.
off_query = '회사 야유회 예산은 1인당 얼마까지 쓸 수 있나요'
print(f'질문: {off_query}\n')
show(search(off_query))

관계없는 질문인데도 **무언가는 돌아옵니다.** 벡터 검색은 "없다"고 답하지 않고 **가장 덜 먼 것**을 주기 때문입니다. 거리값이 앞의 질문보다 확연히 크다는 점이 유일한 단서입니다. 이 성질 때문에 **§6 에서 LLM 에게 "근거에 없으면 모른다고 하라"고 분명히 일러 두어야** 합니다.

### 🖐️ 함께 따라하기

**다른 질문 두 개**로 검색해 결과를 비교합니다.

1. `q_a` 에 `'개인정보 처리방침은 어디에 공개해야 하나요'` 를 담고 `search(q_a, k=3)` 결과를 `show` 로 출력한다.
2. `q_b` 에 `'생성형 AI 서비스에서 이용자가 입력한 대화 내용은 어떻게 처리해야 하나요'` 를 담고 같은 방식으로 출력한다.
3. 두 질문의 **1위 거리**를 각각 뽑아 한 줄로 나란히 출력하고, 어느 질문이 더 또렷한 답을 가졌는지 본다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. `res['documents'][0]` 처럼 `[0]` 을 붙이는 이유는 무엇인가요?
2. 코사인 공간에서 거리 0.20 과 0.45 중 질문에 더 가까운 청크는 어느 쪽인가요? 이유는?

<details><summary>정답 보기</summary>

1. `query` 는 질문을 **여러 개** 받을 수 있어서 결과가 질문별로 한 겹 더 감싸여 있다. 질문이 하나면 그 첫 번째 묶음을 벗겨야 한다.
2. **0.20**. `거리 = 1 - 코사인 유사도` 라 값이 작을수록 유사도가 높다.

</details>

## 5. 질의에서 메타데이터 필터를 자동으로 뽑기

앞 절에서는 질문을 통째로 던져 검색했습니다. 그런데 사용자는 "**처리방침 표준안에서** 알기 쉬운 표현이 뭐라고 하나요" 처럼 **범위를 말로 섞어** 묻습니다. 벡터 검색은 그 '범위'를 조건으로 알아듣지 못하고 뜻이 비슷한 것만 찾습니다. **말로 된 조건을 `where` 필터로 옮기는 다리**가 필요합니다.

지난 시간에 배운 **구조화된 출력**을 그대로 씁니다 — `pydantic` 클래스를 정의해 `client.chat.completions.parse(response_format=클래스)` 에 넘기면, 답이 그 클래스의 객체로 돌아옵니다.

여기서 **두 가지를 반드시 지켜야** 합니다.

1. **허용값을 `Literal` 로 못박고, 시스템 메시지에도 그 값을 적는다.** 그냥 `str` 로 두면 모델이 `'처리방침'`, `'표준안'`, `'개인정보 처리방침 표준(안)'` 처럼 제각각 써서 `where` 가 아무것도 못 찾습니다. 필터 값은 **색인의 메타값과 글자까지 똑같아야** 합니다.
2. **값이 없을 때는 "비워 두어라"고 한다.** "`null` 로 두어라" 라고 적으면 모델이 문자열 `'null'` 을 채워 넣습니다 — 그러면 `문서='null'` 인 필터가 걸려 검색 결과가 0건이 됩니다.

In [ ]:
from typing import Literal, Optional

from pydantic import BaseModel, Field

class QueryFilter(BaseModel):
    """질문에서 뽑아낸 검색 범위. 해당 없으면 값이 없는(None) 채로 둔다."""
    # Literal 로 허용값을 못박는다 -> 색인 메타값과 글자까지 같아야 where 가 맞는다.
    doc_name: Optional[Literal['생성형AI 안내서', '처리방침 표준안']] = Field(
        description='질문이 특정 자료를 가리키면 그 이름, 아니면 값을 비워 둔다')
    pub_year: Optional[Literal[2025, 2026]] = Field(
        description='질문이 특정 발간 연도를 가리키면 그 연도, 아니면 값을 비워 둔다')

# 허용값을 시스템 메시지에도 적어 준다 -- 모델이 스키마만 보고 짐작하지 않게 한다.
FILTER_SYSTEM = (
    '질문에서 검색 범위를 뽑는다. 자료는 두 가지다: '
    "'생성형AI 안내서'(2025년 발간, 생성형 AI 개발·활용과 개인정보), "
    "'처리방침 표준안'(2026년 발간, 개인정보 처리방침 작성). "
    '질문이 어느 자료 또는 어느 연도인지 분명히 가리킬 때만 값을 채우고, '
    '그렇지 않으면 값을 비워 두어라.')

def extract_filter(question):
    """질문 한 문장에서 검색 범위를 뽑는다. OpenAI 를 1회 호출한다."""
    resp = client.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[{'role': 'system', 'content': FILTER_SYSTEM},
                  {'role': 'user', 'content': question}],
        # 같은 질문이면 같은 필터가 나와야 한다 -> 무작위성을 끈다.
        temperature=0,
        response_format=QueryFilter)
    return resp.choices[0].message.parsed

print("필터 추출기 준비 완료")

범위를 **말한 질문**과 **말하지 않은 질문**을 각각 넣어 봅니다. (OpenAI 호출 2회)

In [ ]:
# 자료 이름을 말한 질문과, 아무 범위도 말하지 않은 질문을 하나씩 준비한다.
scoped = '생성형AI 안내서는 개인정보 처리 목적을 어떻게 정하라고 하나요'
plain = '개인정보를 안전하게 관리하려면 무엇을 해야 하나요'

scoped_filter = extract_filter(scoped)
plain_filter = extract_filter(plain)

print(f"[범위를 말한 질문] {scoped}")
print(f"   -> {scoped_filter}")
print(f"\n[범위를 말하지 않은 질문] {plain}")
print(f"   -> {plain_filter}")

범위를 말하지 않은 질문에서는 두 값이 **`None`** 으로 비어 나옵니다. 문자열 `'null'` 이 아니라 진짜 비어 있는 값입니다.

이제 이 결과를 `where` 로 옮깁니다. 여기서 **셋째 함정**을 피해야 합니다.

> ⚠️ `if 값:` 이 아니라 **`if 값 is not None:`** 으로 검사합니다. `if 값:` 은 **0 과 빈 문자열도 거짓**으로 보기 때문에, 연도 같은 숫자 필터를 다룰 때 멀쩡한 값이 조용히 빠집니다.

In [ ]:
def build_where(query_filter):
    """뽑아낸 범위를 ChromaDB 의 where 조건으로 바꾼다. 채워진 값이 없으면 None."""
    conditions = {}
    # 0 이나 빈 문자열도 거짓으로 보는 'if 값:' 대신 'is not None' 으로 검사한다.
    # 뽑아낸 값을 색인의 메타 키('문서'·'발간') 에 그대로 옮겨 담는다.
    if query_filter.doc_name is not None:
        conditions['문서'] = query_filter.doc_name
    if query_filter.pub_year is not None:
        conditions['발간'] = query_filter.pub_year

    if not conditions:
        return None
    # 조건이 둘이면 ChromaDB 는 $and 로 묶어 줘야 알아듣는다.
    if len(conditions) == 1:
        return conditions
    return {'$and': [{key: value} for key, value in conditions.items()]}

scoped_where = build_where(scoped_filter)
print('범위를 말한 질문의 where :', scoped_where)
print('말하지 않은 질문의 where :', build_where(plain_filter))

**필터가 정말 달라지게 만드는지** 확인합니다. **같은 질문**을 필터 없이 한 번, 필터를 걸어 한 번 검색해 비교합니다.

In [ ]:
print(f'질문: {scoped}\n')

print('[필터 없이]')
show(search(scoped, k=3))

print(f'\n[필터: {scoped_where}]')
show(search(scoped, k=3, where=scoped_where))

질문이 **자료 이름을 분명히 말했는데도** 필터가 없으면 다른 자료의 청크가 위로 올라옵니다. 두 안내서가 모두 '개인정보 처리 목적'을 이야기하기 때문에 벡터만으로는 갈라내지 못하는 것입니다. 필터를 걸면 그 청크가 결과에서 빠지고, 질문이 가리킨 자료 안에서만 순위가 매겨집니다.

> 남아 있는 청크의 **거리값 자체는 변하지 않습니다.** 필터는 순위 계산을 바꾸는 게 아니라 **후보를 줄이는** 일이기 때문입니다.

> 반대로, 두 자료가 다루는 주제가 겹치지 않는 질문에서는 필터를 걸어도 **결과가 그대로일 수 있습니다.** 필터는 성능을 올리는 마법이 아니라 **틀린 자료가 섞여 들어올 때 그것을 막는 장치**입니다.

### 🖐️ 함께 따라하기

**연도로 범위를 말한 질문**을 처리해 봅니다. 질문은 **"2026년 표준안은 개인정보 국외이전을 어떻게 적으라고 하나요"** 입니다. (OpenAI 호출 1회)

1. 질문을 `my_q` 에 담고 `extract_filter(my_q)` 로 범위를 뽑아 출력한다.
2. `build_where(...)` 로 `where` 조건을 만들어 출력한다.
3. 필터 없이 `search(my_q, k=3)` 한 결과와, `where` 를 걸어 검색한 결과를 각각 `show` 로 출력한다.
4. 두 결과에 담긴 **문서 이름**이 어떻게 달라졌는지 눈으로 확인한다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. 문서 이름을 `Optional[str]` 로 두지 않고 `Literal[...]` 로 못박는 이유는 무엇인가요?
2. `if query_filter.pub_year:` 라고 쓰면 어떤 값에서 문제가 생기나요?

<details><summary>정답 보기</summary>

1. 필터 값은 색인의 메타값과 **글자까지 똑같아야** 맞는다. 자유 문자열로 두면 모델이 표기를 제각각 만들어 내 `where` 가 아무것도 찾지 못한다.
2. **0**(과 빈 문자열)이 거짓으로 판정돼 조건에서 빠진다. 지금 스키마엔 0이 없지만, 숫자 필터를 다루는 코드에서는 `is not None` 으로 검사하는 습관을 들여야 한다.

</details>

## 6. 근거를 모아 답을 만들고, 출처를 붙인다

이제 마지막 조각입니다. 답만 있으면 학생도 실무자도 그 말을 믿을 근거가 없습니다. **어느 문서 몇 쪽에서 왔는지**가 붙어 있어야 PDF 를 열어 직접 대조할 수 있습니다. RAG 가 LLM 단독보다 나은 진짜 이유가 이것입니다.

**핵심 규칙 하나.** 출처에 쓸 정보는 **컨텍스트 문자열 안에 직접 실어야** 합니다. 쪽 번호를 넣지 않고 프롬프트로만 "쪽을 밝혀라"라고 하면, 모델은 볼 수 없는 것을 요구받아 **쪽 번호를 지어내거나** "알 수 없다"고 답합니다. 모델은 우리가 준 글자만 봅니다.

<img src="images/파이프라인_한바퀴.png" width="960">

In [ ]:
def build_context(hits):
    """검색 결과를 근거 묶음 문자열로 만든다. 문서명과 쪽을 글자로 함께 싣는다."""
    blocks = []
    for _, text, meta, _ in hits:
        # 이 머리줄이 없으면 모델은 쪽 번호를 볼 수 없어 지어내게 된다.
        blocks.append(f"[출처: {meta['문서']} {meta['쪽']}쪽]\n{text}")
    return '\n\n'.join(blocks)

ANSWER_SYSTEM = (
    '너는 개인정보 보호 안내서를 근거로 답하는 도우미다. '
    '아래 근거에 적힌 내용만으로 답하라. '
    '근거에 답이 없으면 "제공된 자료에서 확인할 수 없습니다" 라는 한 문장만 쓰고 출처는 적지 마라. '
    '답을 쓴 경우에는 맨 끝 줄에 사용한 근거를 모두 적어라. '
    '형식은 다음 예시와 글자까지 똑같이 맞춰라 -- 예) 출처: 생성형AI 안내서 33쪽 '
    '(꺾쇠나 따옴표를 붙이지 마라). 근거가 여럿이면 쉼표로 잇되 '
    '항목마다 문서명을 다시 적어라 -- 예) 출처: 생성형AI 안내서 33쪽, 생성형AI 안내서 21쪽 '
    '근거 머리줄에 적힌 문서명과 쪽 번호를 그대로 쓰고, 지어내지 마라.')

def answer(question, context):
    """근거를 실어 답을 만든다. OpenAI 를 1회 호출한다."""
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'system', 'content': ANSWER_SYSTEM},
                  {'role': 'user', 'content': f'[근거]\n{context}\n\n[질문] {question}'}],
        temperature=0,
        max_tokens=600)
    return resp.choices[0].message.content

print("답 생성기 준비 완료")

이제 §4 에서 만든 `search` 로 근거를 찾고, 그대로 답을 만들어 봅니다. (OpenAI 호출 1회)

In [ ]:
rag_question = 'AI 학습에 쓸 데이터를 가명처리할 때 무엇을 확인해야 하나요'

hits = search(rag_question, k=3)                # 1) 검색: 가까운 조각 3개
context_hits = build_context(hits)              # 2) 근거 묶음: 출처 머리줄을 붙여 이어 붙인다

print(f"근거 {len(hits)}조각 · {len(context_hits):,}자")
print("\n----- 근거 앞부분 -----")
print(context_hits[:300])
print("\n----- 답 -----")
print(answer(rag_question, context_hits))       # 3) 생성: 그 근거만 보고 답하게 한다

답 끝에 **문서명과 쪽 번호**가 붙어 나옵니다. `data/원본/` 의 PDF 를 열어 그 쪽으로 가 보면 실제로 그 내용이 있습니다 — 이것이 확인 가능한 답입니다.

### 근거를 얼마나 넣을 것인가

위에서는 **찾은 조각만** 넣었습니다. 조각은 400자 남짓이라, 답에 필요한 문장이 조각 경계 바로 바깥에 있으면 잘려 나갈 수 있습니다. 근거의 범위를 넓히는 방법이 몇 가지 있습니다.

| 방법 | 넣는 것 | 기대 | 대가 |
|---|---|---|---|
| 1) 찾은 조각만 | 검색된 청크 그대로 | 군더더기가 없다 | 경계 밖 내용이 빠진다 |
| 2) 앞뒤 조각까지 | 같은 쪽의 바로 앞·뒤 조각을 더한다 | 끊긴 문맥이 이어진다 | 글자 수가 늘어난다 |
| 3) 그 쪽 전체 | 조각이 속한 쪽의 본문 전체 | 빠지는 내용이 거의 없다 | 관계없는 내용까지 섞이고 비용이 커진다 |

> 여기서는 셋을 **직접 만들어 눈으로 견주는 것**까지만 합니다. 이 셋을 체계적으로 비교하고 각각에 이름을 붙이는 것은 뒤 단원의 일입니다.

글자 수와 답이 실제로 어떻게 달라지는지 재 봅니다. 앞뒤 조각은 **`쪽id-순번`** 으로 지은 청크 id 덕분에 번호만 ±1 해서 찾을 수 있습니다.

In [ ]:
# 청크 id 로 본문을 바로 찾을 수 있게 표를 하나 만들어 둔다.
text_by_id = dict(zip(chunk_ids, chunk_texts))
meta_by_id = dict(zip(chunk_ids, chunk_metas))

def widen_with_neighbors(hits):
    """찾은 조각과 그 바로 앞·뒤 조각까지 근거로 삼는다(같은 쪽 안에서만)."""
    picked = []
    for chunk_id, _, _, _ in hits:
        # id 가 '쪽id-순번' 이라 순번만 ±1 하면 앞뒤 조각이 된다.
        doc_id, order = chunk_id.rsplit('-', 1)
        for step in (-1, 0, 1):
            neighbor = f'{doc_id}-{int(order) + step}'
            # 쪽의 첫/마지막 조각이면 이웃이 없다 -> 있는 것만 담는다.
            if neighbor in text_by_id and neighbor not in picked:
                picked.append(neighbor)

    # 원래 순서대로 정렬해야 글이 이어져 읽힌다.
    blocks = []
    for chunk_id in [cid for cid in chunk_ids if cid in picked]:
        chunk_meta = meta_by_id[chunk_id]
        blocks.append(f"[출처: {chunk_meta['문서']} {chunk_meta['쪽']}쪽]\n{text_by_id[chunk_id]}")
    return '\n\n'.join(blocks)

def widen_to_pages(hits):
    """찾은 조각이 속한 쪽의 본문 전체를 근거로 삼는다."""
    seen, blocks = [], []
    for _, _, meta, _ in hits:
        if meta['doc_id'] in seen:
            continue
        seen.append(meta['doc_id'])
        row = guide[guide['id'] == meta['doc_id']].iloc[0]
        blocks.append(f"[출처: {row['문서']} {row['쪽']}쪽]\n{row['본문']}")
    return '\n\n'.join(blocks)

context_neighbors = widen_with_neighbors(hits)
context_pages = widen_to_pages(hits)

print(f"1) 찾은 조각만   {len(context_hits):>6,}자")
print(f"2) 앞뒤 조각까지 {len(context_neighbors):>6,}자")
print(f"3) 그 쪽 전체    {len(context_pages):>6,}자")

글자 수가 이만큼 차이 납니다. 이 차이는 그대로 **비용과 응답 시간**입니다. 그럼 답은 얼마나 달라질까요. 2)와 3)으로 같은 질문에 답해 봅니다. (OpenAI 호출 2회)

In [ ]:
print('----- 2) 앞뒤 조각까지 -----')
print(answer(rag_question, context_neighbors))

print('\n----- 3) 그 쪽 전체 -----')
print(answer(rag_question, context_pages))

**확실한 것은 글자 수뿐입니다.** 1)에서 3)으로 가며 근거가 두 배 넘게 늘었고, 그 차이는 그대로 **비용과 응답 시간**입니다.

**답이 얼마나 달라졌는지는 여러분이 직접 읽고 판단해야 합니다.** 세 답을 나란히 놓고 이렇게 물어보세요.

- 넓힌 쪽 답에 **1)에 없던 내용**이 실제로 들어왔는가?
- 들어왔다면 그것이 **질문에 답하는 내용**인가, 아니면 그 쪽에 함께 있었을 뿐인 다른 이야기인가?
- 출처로 적힌 쪽이 늘었다면, 그 쪽이 **정말 근거로 쓰인** 것인가?

이 질문들에 "아니오"가 나오면 **늘어난 글자 수만큼 비용만 더 쓴 것**입니다. 근거를 넓히는 일은 **자동으로 좋아지는 일이 아닙니다.**

> 같은 셀을 다시 실행하면 답의 문장이 조금씩 달라집니다. LLM 이라 그렇습니다 — 그래서 이런 비교는 **한 번 보고 단정하지 말고** 여러 질문으로 되풀이해 봐야 합니다. 검색 쪽을 지표로 재는 방법은 다음 시간에 배웁니다.

**고르는 기준**은 이렇습니다.

- 질문이 **좁고 사실 확인**이면 1) 로 충분하다 — 짧고 싸고 빠르다.
- 답이 자꾸 **문장 중간에서 끊긴 듯**하면 2) 로 넓힌다.
- 문서 한 쪽이 하나의 완결된 이야기이고 **빠짐 없는 답**이 중요하면 3) 을 쓴다. 대신 비용을 각오한다.

**자료에 없는 것**을 물었을 때도 확인해 봐야 합니다. §4 에서 봤듯 검색은 무엇이든 돌려주므로, 근거가 엉뚱해도 모델이 그럴듯하게 지어낼 위험이 있습니다. (OpenAI 호출 1회)

In [ ]:
# 이 안내서에 있을 리 없는 질문 -- 시스템 메시지의 '근거에 없으면 모른다' 지시가 실제로 먹는지 본다.
print(f'질문: {off_query}\n')
off_hits = search(off_query, k=3)
print(answer(off_query, build_context(off_hits)))

"제공된 자료에서 확인할 수 없습니다" 로 답합니다. **이 문장이 나오도록 시스템 메시지에 미리 못박아 둔** 덕분입니다. 이 한 줄이 없으면 모델은 엉뚱한 근거를 억지로 엮어 답을 만들어 냅니다.

### 🖐️ 함께 따라하기

지금까지 만든 조각을 **한 함수로 이어** 파이프라인을 완성합니다. 질문은 **"처리방침 표준안은 개인정보 파기에 대해 무엇을 적으라고 하나요"** 입니다. (OpenAI 호출 2회)

1. `ask(question, k=3)` 함수를 만든다. 안에서 차례로:
   `extract_filter` 로 범위를 뽑고 → `build_where` 로 조건을 만들고 → `search(question, k, where=...)` 로 찾고 → `build_context` 로 근거를 만들고 → `answer` 로 답을 만들어 **돌려준다**.
2. 위 질문으로 `ask` 를 불러 답을 출력한다.
3. 답에 적힌 출처의 **쪽 번호**가 검색 결과의 쪽과 같은지 눈으로 확인한다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. 쪽 번호를 컨텍스트 문자열에 넣지 않고 시스템 메시지로만 "쪽을 밝혀라"라고 하면 무슨 일이 벌어지나요?
2. 짧은 사실 확인 질문에 굳이 '그 쪽 전체'를 근거로 넣지 않는 이유 두 가지를 말해 보세요.

<details><summary>정답 보기</summary>

1. 모델은 우리가 준 글자만 본다. 쪽 번호가 근거 안에 없으면 **지어내거나** 알 수 없다고 답한다.
2. 글자 수가 늘어 **비용과 응답 시간**이 커지고, 질문과 상관없는 내용이 섞여 **답이 산만해진다**.

</details>

## 이번 강의 정리

| 단계 | 한 일 | 도구·규칙 |
|---|---|---|
| 파싱 | PDF → 쪽 단위 마크다운 | `pymupdf4llm.to_markdown(..., page_chunks=True)` · 쪽은 `metadata['page_number']` |
| 도구 선택 | 마크다운이냐 순수 텍스트냐 | 표·제목이 뜻을 나르면 마크다운, 줄글·대량이면 `get_text` |
| 청킹 | 쪽을 조각으로 | 고정 · 겹침 · 문단 — **데이터를 보고 고른다**(이 문서는 문단) |
| 색인 | 조각을 임베딩해 저장 | ChromaDB `PersistentClient` + **지문**으로 재사용 |
| 검색 | 질문 → 조각 | 본문 · 메타 · **거리**를 함께 읽는다(`거리 = 1 - 유사도`) |
| 필터 | 말로 된 범위 → `where` | `Literal` 로 값 고정 · 없으면 **비워 두기** · `is not None` 으로 검사 |
| 생성 | 근거 → 답 + 출처 | **문서명·쪽을 컨텍스트에 실어야** 출처가 진짜가 된다 |

가장 중요한 습관 하나만 남긴다면: **모든 단계의 선택을 숫자로 확인하는 것**입니다. 오늘 우리는 도구도, 청킹 전략도, 근거 범위도 "좋다더라"가 아니라 직접 재 본 값으로 골랐습니다.

## ⏭️ 예고 — 다음 시간: 검색 품질을 재고 개선하기

오늘 만든 파이프라인은 **잘 도는지**는 봤지만 **얼마나 잘 찾는지**는 재지 않았습니다. 다음 시간엔 정답이 붙은 질문 묶음을 만들어 검색 품질을 **지표로** 재고, 그 숫자를 근거로 파이프라인을 한 군데 개선합니다.